In [16]:
# ============================================================
# 서울시 역사마스터 ↔ CELL 250m 매핑 최종 코드
# - station -> nearest cell
# - cell -> nearest station
# - 거리 요약 / outlier 확인
# ============================================================

import os
import pandas as pd
import numpy as np
from pyproj import Transformer
from scipy.spatial import cKDTree

# =========================
# 0. 경로 설정
# =========================

cell_csv_path = r"C:\Users\82108\Desktop\새 폴더 (8)\CELL_250x250(UTMK_epsg 5179).csv"
station_csv_path = r"C:\Users\82108\Desktop\새 폴더 (8)\서울시 역사마스터 정보.csv"

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_seoul_master_final"
os.makedirs(output_dir, exist_ok=True)

station_to_cell_path = os.path.join(output_dir, "station_to_nearest_cell.csv")
cell_to_station_path = os.path.join(output_dir, "cell_to_nearest_station.csv")
station_outlier_path = os.path.join(output_dir, "station_mapping_outliers_over_2km.csv")
cell_outlier_path = os.path.join(output_dir, "cell_mapping_outliers_over_5km.csv")


# =========================
# 1. CSV 안전 읽기
# =========================

def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / encoding={enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")


station_df = read_csv_safe(station_csv_path)
cell_df = read_csv_safe(cell_csv_path)

print("\n[STATION RAW]")
print(station_df.head())
print(station_df.columns)

print("\n[CELL RAW]")
print(cell_df.head())
print(cell_df.columns)


# =========================
# 2. 컬럼 정리
# =========================

station_df = station_df.rename(columns={
    "역사_ID": "STATION_ID",
    "역사명": "station_name",
    "호선": "line",
    "위도": "lat",
    "경도": "lon"
})

cell_df = cell_df.rename(columns={
    "cell_id": "CELL_ID",
    "Cell_ID": "CELL_ID",
    "x": "CELL_X",
    "y": "CELL_Y",
    "X": "CELL_X",
    "Y": "CELL_Y"
})

station_df = station_df[["STATION_ID", "station_name", "line", "lat", "lon"]].copy()
cell_df = cell_df[["CELL_ID", "CELL_X", "CELL_Y"]].copy()

station_df["lat"] = pd.to_numeric(station_df["lat"], errors="coerce")
station_df["lon"] = pd.to_numeric(station_df["lon"], errors="coerce")
cell_df["CELL_X"] = pd.to_numeric(cell_df["CELL_X"], errors="coerce")
cell_df["CELL_Y"] = pd.to_numeric(cell_df["CELL_Y"], errors="coerce")

station_df = station_df.dropna(subset=["lat", "lon"])
cell_df = cell_df.dropna(subset=["CELL_X", "CELL_Y"])

station_df = station_df.drop_duplicates(subset=["STATION_ID", "station_name", "line", "lat", "lon"])
cell_df = cell_df.drop_duplicates(subset=["CELL_ID"])

print("\n[STATION CLEAN]")
print(station_df.shape)
print(station_df.head())

print("\n[CELL CLEAN]")
print(cell_df.shape)
print(cell_df.head())


# =========================
# 3. 역 좌표 WGS84 → EPSG:5179
# =========================

transformer = Transformer.from_crs("EPSG:4326", "EPSG:5179", always_xy=True)

station_df["station_x_5179"], station_df["station_y_5179"] = transformer.transform(
    station_df["lon"].values,
    station_df["lat"].values
)

print("\n[STATION COORD CHECK]")
print(station_df[["station_x_5179", "station_y_5179"]].describe())


# =========================
# 4. KDTree 준비
# =========================

cell_points = cell_df[["CELL_X", "CELL_Y"]].to_numpy()
station_points = station_df[["station_x_5179", "station_y_5179"]].to_numpy()

cell_tree = cKDTree(cell_points)
station_tree = cKDTree(station_points)


# ============================================================
# 5. station -> nearest cell 매핑
# ============================================================

station_dist, station_idx = cell_tree.query(station_points, k=1)

matched_cells = cell_df.iloc[station_idx].reset_index(drop=True)

station_to_cell = station_df.reset_index(drop=True).copy()
station_to_cell["matched_CELL_ID"] = matched_cells["CELL_ID"].values
station_to_cell["matched_CELL_X"] = matched_cells["CELL_X"].values
station_to_cell["matched_CELL_Y"] = matched_cells["CELL_Y"].values
station_to_cell["distance_m"] = station_dist

station_to_cell["valid_within_500m"] = station_to_cell["distance_m"] <= 500
station_to_cell["valid_within_1km"] = station_to_cell["distance_m"] <= 1000
station_to_cell["valid_within_2km"] = station_to_cell["distance_m"] <= 2000

station_to_cell.to_csv(station_to_cell_path, index=False, encoding="utf-8-sig")

station_outliers = station_to_cell[station_to_cell["distance_m"] > 2000].copy()
station_outliers.to_csv(station_outlier_path, index=False, encoding="utf-8-sig")


# ============================================================
# 6. cell -> nearest station 매핑
# ============================================================

cell_dist, cell_idx = station_tree.query(cell_points, k=1)

nearest_stations = station_df.iloc[cell_idx].reset_index(drop=True)

cell_to_station = cell_df.reset_index(drop=True).copy()
cell_to_station["STATION_ID"] = nearest_stations["STATION_ID"].values
cell_to_station["nearest_station"] = nearest_stations["station_name"].values
cell_to_station["nearest_line"] = nearest_stations["line"].values
cell_to_station["station_lat"] = nearest_stations["lat"].values
cell_to_station["station_lon"] = nearest_stations["lon"].values
cell_to_station["station_x_5179"] = nearest_stations["station_x_5179"].values
cell_to_station["station_y_5179"] = nearest_stations["station_y_5179"].values
cell_to_station["nearest_station_distance_m"] = cell_dist

cell_to_station["station_access_250m"] = cell_to_station["nearest_station_distance_m"] <= 250
cell_to_station["station_access_500m"] = cell_to_station["nearest_station_distance_m"] <= 500
cell_to_station["station_access_1km"] = cell_to_station["nearest_station_distance_m"] <= 1000
cell_to_station["station_access_2km"] = cell_to_station["nearest_station_distance_m"] <= 2000
cell_to_station["station_access_5km"] = cell_to_station["nearest_station_distance_m"] <= 5000

cell_to_station["station_accessibility_score"] = 1 / (
    cell_to_station["nearest_station_distance_m"] + 1
)

cell_to_station["is_station_area_5km"] = cell_to_station["nearest_station_distance_m"] <= 5000

cell_to_station["valid_nearest_station"] = cell_to_station["nearest_station"]
cell_to_station.loc[
    ~cell_to_station["is_station_area_5km"],
    "valid_nearest_station"
] = np.nan

cell_to_station.to_csv(cell_to_station_path, index=False, encoding="utf-8-sig")

cell_outliers = cell_to_station[cell_to_station["nearest_station_distance_m"] > 5000].copy()
cell_outliers.to_csv(cell_outlier_path, index=False, encoding="utf-8-sig")


# ============================================================
# 7. 결과 확인
# ============================================================

print("\n============================================================")
print("DONE")
print("============================================================")

print("\n저장 파일")
print("station -> cell:", station_to_cell_path)
print("cell -> station:", cell_to_station_path)
print("station outliers:", station_outlier_path)
print("cell outliers:", cell_outlier_path)

print("\n[STATION -> CELL HEAD]")
print(station_to_cell.head())

print("\n[STATION -> CELL DISTANCE]")
print(station_to_cell["distance_m"].describe())

print("\n[STATION -> CELL VALID RATIO]")
print(station_to_cell[[
    "valid_within_500m",
    "valid_within_1km",
    "valid_within_2km"
]].mean())

print("\n[STATION OUTLIERS > 2km]")
print(station_outliers[[
    "STATION_ID",
    "station_name",
    "line",
    "lat",
    "lon",
    "matched_CELL_ID",
    "distance_m"
]].sort_values("distance_m", ascending=False).head(30))

print("\n[CELL -> STATION HEAD]")
print(cell_to_station.head())

print("\n[CELL -> STATION DISTANCE]")
print(cell_to_station["nearest_station_distance_m"].describe())

print("\n[CELL ACCESS RATIO]")
print(cell_to_station[[
    "station_access_250m",
    "station_access_500m",
    "station_access_1km",
    "station_access_2km",
    "station_access_5km",
    "is_station_area_5km"
]].mean())

print("\n[CELL OUTLIERS > 5km]")
print(cell_outliers[[
    "CELL_ID",
    "CELL_X",
    "CELL_Y",
    "nearest_station",
    "nearest_line",
    "nearest_station_distance_m"
]].sort_values("nearest_station_distance_m", ascending=False).head(30))

print("\n[UNIQUE STATION COUNT]")
print("station rows:", len(station_to_cell))
print("unique STATION_ID:", station_to_cell["STATION_ID"].nunique())
print("unique station names:", station_to_cell["station_name"].nunique())

print("\n[TOP MAPPED STATIONS BY CELL COUNT]")
station_cell_count = (
    cell_to_station
    .groupby(["STATION_ID", "nearest_station", "nearest_line"], as_index=False)
    .agg(
        cell_count=("CELL_ID", "count"),
        mean_distance=("nearest_station_distance_m", "mean"),
        min_distance=("nearest_station_distance_m", "min"),
        max_distance=("nearest_station_distance_m", "max"),
        within_1km=("station_access_1km", "sum"),
        within_5km=("station_access_5km", "sum")
    )
    .sort_values("cell_count", ascending=False)
)

print(station_cell_count.head(30))

station_cell_count.to_csv(
    os.path.join(output_dir, "station_cell_count_summary.csv"),
    index=False,
    encoding="utf-8-sig"
)

read success: 서울시 역사마스터 정보.csv / encoding=cp949
read success: CELL_250x250(UTMK_epsg 5179).csv / encoding=utf-8-sig

[STATION RAW]
   역사_ID 역사명          호선        위도         경도
0   9010  동탄  수도권 광역급행철도  37.20034  127.09569
1   9009  구성  수도권 광역급행철도  37.29913  127.10389
2   9008  성남  수도권 광역급행철도  37.39467  127.12058
3   9007  수서  수도권 광역급행철도  37.48637  127.10161
4   9006  삼성  수도권 광역급행철도  37.50887  127.06324
Index(['역사_ID', '역사명', '호선', '위도', '경도'], dtype='object')

[CELL RAW]
      CELL_ID  CELL_X   CELL_Y
0  나사99757775  899875  1977875
1  다사18001325  918125  1913375
2  다사31254225  931375  1942375
3  다사33752875  933875  1928875
4  다바40009800  940125  1898125
Index(['CELL_ID', 'CELL_X', 'CELL_Y'], dtype='object')

[STATION CLEAN]
(783, 5)
   STATION_ID station_name        line       lat        lon
0        9010           동탄  수도권 광역급행철도  37.20034  127.09569
1        9009           구성  수도권 광역급행철도  37.29913  127.10389
2        9008           성남  수도권 광역급행철도  37.39467  127.12058
3        9007   

In [1]:
# ============================================================
# HIGH-CONFIDENCE STATION MATCH PIPELINE
# 서울교통공사_역명 지하철역 검색.csv = 매칭용 dictionary
# 매칭되는 STATION_ID만 사용
# unmatched는 버림
# ============================================================

import os
import pandas as pd
import numpy as np

# =========================
# 0. 경로
# =========================

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\research_high_confidence"
os.makedirs(output_dir, exist_ok=True)

# 매칭용 dictionary
dict_path = r"C:\Users\82108\Desktop\새 폴더 (8)\서울교통공사_역명 지하철역 검색.csv"

# 최종 cell-station mapping
mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv"

# mobility 1개 테스트
mobility_path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

# station activity 데이터
station_30min_path = r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output\TBDM_TRANSIT_STAT_SUBWAY_20170102_station_stats.parquet"

station_daily_path = r"C:\Users\82108\Desktop\새 폴더 (8)\od_output\od_output\202002_TBDM_TRNSIT_OD_SUBWAY_20191001_station.parquet"

# 저장 파일
dict_clean_path = os.path.join(output_dir, "station_dictionary_high_confidence.csv")
mapping_hc_path = os.path.join(output_dir, "cell_station_mapping_high_confidence.csv")
final_path = os.path.join(output_dir, "TEST_high_confidence_final_dataset.csv")
top_hidden_path = os.path.join(output_dir, "TEST_high_confidence_top_hidden.csv")
station_summary_path = os.path.join(output_dir, "TEST_high_confidence_station_summary.csv")


# =========================
# 1. 유틸
# =========================

def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")


def normalize_minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    if s.max() == s.min():
        return pd.Series(0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())


def clean_station_name(s):
    return (
        s.astype(str)
        .str.replace("'", "", regex=False)
        .str.strip()
        .str.replace(" ", "", regex=False)
    )


# =========================
# 2. dictionary 로드
# =========================

station_dict = read_csv_safe(dict_path)

print("\n[DICT RAW]")
print(station_dict.shape)
print(station_dict.columns.tolist())
print(station_dict.head())

station_dict = station_dict.rename(columns={
    "전철역코드": "subway_station_code",
    "전철역명": "station_name",
    "호선": "dict_line",
    "외부코드": "external_code"
})

station_dict["station_name_clean"] = clean_station_name(station_dict["station_name"])
station_dict["external_code_raw"] = station_dict["external_code"].astype(str).str.strip()

# 외부코드가 숫자인 것만 실제 station activity ID 후보로 사용
station_dict["STATS_STATION_ID"] = pd.to_numeric(
    station_dict["external_code_raw"],
    errors="coerce"
)

station_dict = station_dict.dropna(subset=["STATS_STATION_ID"]).copy()
station_dict["STATS_STATION_ID"] = station_dict["STATS_STATION_ID"].astype(int)

station_dict = station_dict[[
    "STATS_STATION_ID",
    "station_name",
    "station_name_clean",
    "dict_line",
    "subway_station_code",
    "external_code_raw"
]].drop_duplicates()

station_dict.to_csv(dict_clean_path, index=False, encoding="utf-8-sig")

print("\n[DICT CLEAN]")
print(station_dict.shape)
print(station_dict.head())
print("saved:", dict_clean_path)


# =========================
# 3. station activity 로드
# =========================

station_30 = pd.read_parquet(station_30min_path)
station_daily = pd.read_parquet(station_daily_path)

station_30["STATION_ID"] = pd.to_numeric(station_30["STATION_ID"], errors="coerce").astype("Int64")
station_daily["STATION_ID"] = pd.to_numeric(station_daily["STATION_ID"], errors="coerce").astype("Int64")

station_30_ids = set(station_30["STATION_ID"].dropna().astype(int).unique())
station_daily_ids = set(station_daily["STATION_ID"].dropna().astype(int).unique())
activity_ids = station_30_ids | station_daily_ids

print("\n[ACTIVITY ID CHECK]")
print("30min station ids:", len(station_30_ids))
print("daily station ids:", len(station_daily_ids))
print("union activity ids:", len(activity_ids))

# dictionary 중 activity에 실제 존재하는 ID만 retained
station_dict_valid = station_dict[
    station_dict["STATS_STATION_ID"].isin(activity_ids)
].copy()

print("\n[DICT VALID IN ACTIVITY]")
print(station_dict_valid.shape)
print(station_dict_valid.head())


# =========================
# 4. mapping 로드 + dictionary로 high-confidence 매칭
# =========================

mapping = read_csv_safe(mapping_path)

print("\n[MAPPING RAW]")
print(mapping.shape)
print(mapping.columns.tolist())
print(mapping.head())

mapping = mapping.rename(columns={
    "STATION_ID": "MASTER_STATION_ID"
})

mapping["nearest_station_clean"] = clean_station_name(mapping["nearest_station"])

# 역명 기준 dictionary 매칭
# 매칭 안 되면 버림
mapping_hc = mapping.merge(
    station_dict_valid,
    left_on="nearest_station_clean",
    right_on="station_name_clean",
    how="inner"
)

# 중복 대응 제거:
# 같은 CELL_ID가 여러 역 ID로 붙으면 nearest_line과 dict_line 정보가 애매할 수 있으므로
# 일단 동일 cell 중 첫 번째만 사용. 필요하면 나중에 line rule 추가.
mapping_hc = (
    mapping_hc
    .sort_values(["CELL_ID", "nearest_station_distance_m", "STATS_STATION_ID"])
    .drop_duplicates(subset=["CELL_ID"])
    .copy()
)

mapping_hc.to_csv(mapping_hc_path, index=False, encoding="utf-8-sig")

print("\n[HIGH CONFIDENCE MAPPING]")
print("original valid mapping rows:", len(mapping))
print("high-confidence rows:", len(mapping_hc))
print("kept ratio:", len(mapping_hc) / len(mapping))
print("unique STATS_STATION_ID:", mapping_hc["STATS_STATION_ID"].nunique())
print(mapping_hc[[
    "CELL_ID",
    "nearest_station",
    "nearest_line",
    "MASTER_STATION_ID",
    "STATS_STATION_ID",
    "station_name",
    "dict_line",
    "nearest_station_distance_m"
]].head(30))
print("saved:", mapping_hc_path)


# =========================
# 5. mobility 1개 로드 + 집계
# =========================

mobility = pd.read_parquet(mobility_path)

print("\n[MOBILITY RAW]")
print(mobility.shape)
print(mobility.columns.tolist())
print(mobility.head())

needed = ["datetime", "CELL_ID_BASE", "out_cnt", "in_cnt", "net_cnt"]
missing = [c for c in needed if c not in mobility.columns]
if missing:
    raise ValueError(f"mobility 컬럼 없음: {missing}")

mobility = mobility[needed].copy()

mobility = mobility.rename(columns={
    "CELL_ID_BASE": "CELL_ID",
    "out_cnt": "mobility_out_cnt",
    "in_cnt": "mobility_in_cnt",
    "net_cnt": "mobility_net_cnt"
})

for c in ["mobility_out_cnt", "mobility_in_cnt", "mobility_net_cnt"]:
    mobility[c] = pd.to_numeric(mobility[c], errors="coerce").fillna(0)

cell_mobility = (
    mobility
    .groupby("CELL_ID", as_index=False)
    .agg(
        mobility_out_sum=("mobility_out_cnt", "sum"),
        mobility_in_sum=("mobility_in_cnt", "sum"),
        mobility_net_sum=("mobility_net_cnt", "sum"),
        mobility_out_mean=("mobility_out_cnt", "mean"),
        mobility_in_mean=("mobility_in_cnt", "mean"),
        mobility_record_count=("datetime", "count")
    )
)

cell_mobility["actual_total_flow"] = (
    cell_mobility["mobility_out_sum"] + cell_mobility["mobility_in_sum"]
)

cell_mobility["actual_abs_net_flow"] = cell_mobility["mobility_net_sum"].abs()

print("\n[CELL MOBILITY]")
print(cell_mobility.shape)
print(cell_mobility.head())


# =========================
# 6. station activity 집계
# =========================

for col in ["od_out_cnt", "od_in_cnt", "od_net_cnt"]:
    station_30[col] = pd.to_numeric(station_30[col], errors="coerce").fillna(0)
    station_daily[col] = pd.to_numeric(station_daily[col], errors="coerce").fillna(0)

station_30_agg = (
    station_30
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_30_out_sum=("od_out_cnt", "sum"),
        station_30_in_sum=("od_in_cnt", "sum"),
        station_30_net_sum=("od_net_cnt", "sum"),
        station_30_record_count=("datetime", "count")
    )
)

station_30_agg["station_30_total_activity"] = (
    station_30_agg["station_30_out_sum"] + station_30_agg["station_30_in_sum"]
)

station_daily_agg = (
    station_daily
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_daily_out_sum=("od_out_cnt", "sum"),
        station_daily_in_sum=("od_in_cnt", "sum"),
        station_daily_net_sum=("od_net_cnt", "sum"),
        station_daily_record_count=("datetime", "count")
    )
)

station_daily_agg["station_daily_total_activity"] = (
    station_daily_agg["station_daily_out_sum"] + station_daily_agg["station_daily_in_sum"]
)

station_activity = station_30_agg.merge(
    station_daily_agg,
    on="STATS_STATION_ID",
    how="outer"
).fillna(0)

station_activity["station_total_activity"] = (
    station_activity["station_30_total_activity"] +
    station_activity["station_daily_total_activity"]
)

station_activity["station_total_in"] = (
    station_activity["station_30_in_sum"] +
    station_activity["station_daily_in_sum"]
)

station_activity["station_total_out"] = (
    station_activity["station_30_out_sum"] +
    station_activity["station_daily_out_sum"]
)

station_activity["station_total_net"] = (
    station_activity["station_total_in"] -
    station_activity["station_total_out"]
)

print("\n[STATION ACTIVITY]")
print(station_activity.shape)
print(station_activity.head())


# =========================
# 7. 최종 merge
# =========================

final = cell_mobility.merge(
    mapping_hc,
    on="CELL_ID",
    how="inner"
)

final = final.merge(
    station_activity,
    on="STATS_STATION_ID",
    how="left"
)

activity_cols = [c for c in final.columns if c.startswith("station_")]
for c in activity_cols:
    if final[c].dtype != "object":
        final[c] = final[c].fillna(0)

print("\n[FINAL MERGE CHECK]")
print("mobility cells:", cell_mobility["CELL_ID"].nunique())
print("high-confidence mapping cells:", mapping_hc["CELL_ID"].nunique())
print("final cells:", final["CELL_ID"].nunique())
print("final rows:", len(final))
print("activity matched rows:", (final["station_total_activity"] > 0).sum())
print("activity zero rows:", (final["station_total_activity"] == 0).sum())


# =========================
# 8. 연구 지표
# =========================

final["actual_total_flow_norm"] = normalize_minmax(final["actual_total_flow"])
final["station_activity_norm"] = normalize_minmax(final["station_total_activity"])
final["accessibility_norm"] = normalize_minmax(final["station_accessibility_score"])
final["distance_norm"] = normalize_minmax(final["nearest_station_distance_m"])

final["hidden_demand_score"] = (
    final["actual_total_flow_norm"] *
    (1 - final["accessibility_norm"])
)

final["subway_representation_gap"] = (
    final["actual_total_flow_norm"] -
    final["station_activity_norm"]
)

final["subway_overrepresentation_score"] = (
    final["station_activity_norm"] *
    (1 - final["actual_total_flow_norm"])
)

final["station_area_mobility_score"] = (
    final["actual_total_flow_norm"] *
    final["accessibility_norm"]
)

q_hidden = final["hidden_demand_score"].quantile(0.90)
q_gap = final["subway_representation_gap"].quantile(0.90)
q_station_area = final["station_area_mobility_score"].quantile(0.90)

conditions = [
    final["hidden_demand_score"] >= q_hidden,
    final["subway_representation_gap"] >= q_gap,
    final["station_area_mobility_score"] >= q_station_area
]

choices = [
    "hidden_demand_area",
    "subway_under_represented",
    "station_area_high_mobility"
]

final["area_type"] = np.select(conditions, choices, default="normal")


# =========================
# 9. 요약 저장
# =========================

top_hidden = (
    final
    .sort_values("hidden_demand_score", ascending=False)
    .head(200)
)

station_summary = (
    final
    .groupby(["STATS_STATION_ID", "station_name", "dict_line"], as_index=False)
    .agg(
        cell_count=("CELL_ID", "count"),
        total_actual_flow=("actual_total_flow", "sum"),
        mean_actual_flow=("actual_total_flow", "mean"),
        mean_distance_m=("nearest_station_distance_m", "mean"),
        station_total_activity=("station_total_activity", "mean"),
        mean_hidden_demand=("hidden_demand_score", "mean"),
        max_hidden_demand=("hidden_demand_score", "max"),
        mean_representation_gap=("subway_representation_gap", "mean"),
        hidden_cell_count=("area_type", lambda x: (x == "hidden_demand_area").sum())
    )
)

station_summary["hidden_cell_ratio"] = (
    station_summary["hidden_cell_count"] / station_summary["cell_count"]
)

station_summary = station_summary.sort_values(
    ["hidden_cell_count", "total_actual_flow"],
    ascending=False
)

final.to_csv(final_path, index=False, encoding="utf-8-sig")
top_hidden.to_csv(top_hidden_path, index=False, encoding="utf-8-sig")
station_summary.to_csv(station_summary_path, index=False, encoding="utf-8-sig")


# =========================
# 10. 결과 확인
# =========================

print("\n================================================")
print("DONE")
print("================================================")
print("dict clean:", dict_clean_path)
print("mapping high confidence:", mapping_hc_path)
print("final:", final_path)
print("top hidden:", top_hidden_path)
print("station summary:", station_summary_path)

print("\n[FINAL SHAPE]")
print(final.shape)

print("\n[KEY METRICS]")
print(final[[
    "actual_total_flow",
    "nearest_station_distance_m",
    "station_total_activity",
    "actual_total_flow_norm",
    "station_activity_norm",
    "accessibility_norm",
    "hidden_demand_score",
    "subway_representation_gap",
    "subway_overrepresentation_score",
    "station_area_mobility_score"
]].describe())

print("\n[AREA TYPE COUNTS]")
print(final["area_type"].value_counts())

print("\n[TOP HIDDEN]")
print(top_hidden[[
    "CELL_ID",
    "STATS_STATION_ID",
    "station_name",
    "dict_line",
    "nearest_station",
    "nearest_line",
    "actual_total_flow",
    "nearest_station_distance_m",
    "station_total_activity",
    "hidden_demand_score",
    "subway_representation_gap",
    "area_type"
]].head(30))

print("\n[STATION SUMMARY TOP]")
print(station_summary.head(30))

read success: 서울교통공사_역명 지하철역 검색.csv / cp949

[DICT RAW]
(799, 4)
['전철역코드', '전철역명', '호선', '외부코드']
  전철역코드       전철역명    호선  외부코드
0  101C         회기   경의선  K118
1  0205  동대문역사문화공원  02호선   205
2  0206         신당  02호선   206
3  0207       상왕십리  02호선   207
4  0208        왕십리  02호선   208

[DICT CLEAN]
(426, 6)
   STATS_STATION_ID station_name station_name_clean dict_line  \
1               205    동대문역사문화공원          동대문역사문화공원      02호선   
2               206           신당                 신당      02호선   
3               207         상왕십리               상왕십리      02호선   
4               208          왕십리                왕십리      02호선   
5               209          한양대                한양대      02호선   

  subway_station_code external_code_raw  
1                0205               205  
2                0206               206  
3                0207               207  
4                0208               208  
5                0209               209  
saved: C:\Users\82108\Desktop\새 폴더 (8)\research_hig

In [ ]:
import pandas as pd

# CELL master
cell_master = pd.read_csv(
    r"C:\Users\82108\Desktop\새 폴더 (8)\CELL_250x250(UTMK_epsg 5179).csv",
    encoding="utf-8-sig"
)

# mobility parquet
mobility = pd.read_parquet(
    r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"
)

print("\n[CELL MASTER]")
print(cell_master.head())

print("\n[MOBILITY]")
print(mobility.head())

# 예시 index 확인
idx = 26

print("\n[INDEX TEST]")
print(cell_master.iloc[idx])


[MAPPING]
(207205, 19)
['CELL_ID', 'CELL_X', 'CELL_Y', 'STATION_ID', 'nearest_station', 'nearest_line', 'station_lat', 'station_lon', 'station_x_5179', 'station_y_5179', 'nearest_station_distance_m', 'station_access_250m', 'station_access_500m', 'station_access_1km', 'station_access_2km', 'station_access_5km', 'station_accessibility_score', 'is_station_area_5km', 'valid_nearest_station']
      CELL_ID  CELL_X   CELL_Y  STATION_ID nearest_station nearest_line  \
0  나사99757775  899875  1977875        4920              양촌       김포골드라인   
1  다사18001325  918125  1913375        3135          지식정보단지        인천1호선   
2  다사31254225  931375  1942375        3121              동수        인천1호선   
3  다사33752875  933875  1928875        1761              정왕          안산선   
4  다바40009800  940125  1898125        1875              어천          수인선   

   station_lat  station_lon  station_x_5179  station_y_5179  \
0    37.642379   126.614309   921861.795997    1.960691e+06   
1    37.378384   126.645168   9

In [ ]:
import os
import pandas as pd
import numpy as np

# ============================================================
# 0. 경로
# ============================================================

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\station_level_research_test"
os.makedirs(output_dir, exist_ok=True)

cell_master_path = r"C:\Users\82108\Desktop\새 폴더 (8)\CELL_250x250(UTMK_epsg 5179).csv"
mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv"
station_dict_path = r"C:\Users\82108\Desktop\새 폴더 (8)\서울교통공사_역명 지하철역 검색.csv"

mobility_path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

station_30min_path = r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output\TBDM_TRANSIT_STAT_SUBWAY_20170102_station_stats.parquet"
station_daily_path = r"C:\Users\82108\Desktop\새 폴더 (8)\od_output\od_output\202002_TBDM_TRNSIT_OD_SUBWAY_20191001_station.parquet"

cell_mobility_output = os.path.join(output_dir, "01_cell_mobility_restored.csv")
station_mobility_output = os.path.join(output_dir, "02_station_surrounding_mobility.csv")
final_output = os.path.join(output_dir, "03_station_level_final_gap_analysis.csv")
top_hidden_output = os.path.join(output_dir, "04_top_hidden_demand_stations.csv")

# ============================================================
# 1. 유틸
# ============================================================

def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")

def normalize_minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    if s.max() == s.min():
        return pd.Series(0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())

def clean_name(s):
    return (
        s.astype(str)
        .str.replace("'", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )

# ============================================================
# 2. CELL master 로드
# ============================================================

cell_master = read_csv_safe(cell_master_path)
cell_master = cell_master[["CELL_ID", "CELL_X", "CELL_Y"]].copy()

print("\n[CELL MASTER]")
print(cell_master.shape)
print(cell_master.head())

# ============================================================
# 3. mobility 로드 + CELL_ID_BASE → 실제 CELL_ID 복원
# ============================================================

mobility = pd.read_parquet(mobility_path)

print("\n[MOBILITY RAW]")
print(mobility.shape)
print(mobility.head())

mobility["CELL_INDEX"] = pd.to_numeric(mobility["CELL_ID_BASE"], errors="coerce")
mobility = mobility.dropna(subset=["CELL_INDEX"]).copy()
mobility["CELL_INDEX"] = mobility["CELL_INDEX"].astype(int)

valid_idx = (
    (mobility["CELL_INDEX"] >= 0) &
    (mobility["CELL_INDEX"] < len(cell_master))
)

print("\n[CELL INDEX CHECK]")
print("valid ratio:", valid_idx.mean())
print("min:", mobility["CELL_INDEX"].min())
print("max:", mobility["CELL_INDEX"].max())
print("cell master rows:", len(cell_master))

mobility = mobility[valid_idx].copy().reset_index(drop=True)

matched_cell = cell_master.iloc[mobility["CELL_INDEX"].values].reset_index(drop=True)

mobility["CELL_ID"] = matched_cell["CELL_ID"].values
mobility["CELL_X"] = matched_cell["CELL_X"].values
mobility["CELL_Y"] = matched_cell["CELL_Y"].values

for c in ["out_cnt", "in_cnt", "net_cnt"]:
    mobility[c] = pd.to_numeric(mobility[c], errors="coerce").fillna(0)

# CELL 단위 생활이동량 집계
cell_mobility = (
    mobility
    .groupby("CELL_ID", as_index=False)
    .agg(
        CELL_X=("CELL_X", "first"),
        CELL_Y=("CELL_Y", "first"),
        mobility_out_sum=("out_cnt", "sum"),
        mobility_in_sum=("in_cnt", "sum"),
        mobility_net_sum=("net_cnt", "sum"),
        mobility_out_mean=("out_cnt", "mean"),
        mobility_in_mean=("in_cnt", "mean"),
        mobility_record_count=("datetime", "count")
    )
)

cell_mobility["actual_total_flow"] = (
    cell_mobility["mobility_out_sum"] + cell_mobility["mobility_in_sum"]
)

cell_mobility["actual_abs_net_flow"] = cell_mobility["mobility_net_sum"].abs()

cell_mobility.to_csv(cell_mobility_output, index=False, encoding="utf-8-sig")

print("\n[CELL MOBILITY RESTORED]")
print(cell_mobility.shape)
print(cell_mobility.head())

# ============================================================
# 4. cell → nearest station mapping 로드
# ============================================================

mapping = read_csv_safe(mapping_path)

mapping = mapping.rename(columns={
    "STATION_ID": "MASTER_STATION_ID"
})

mapping["nearest_station_clean"] = clean_name(mapping["nearest_station"])

print("\n[MAPPING]")
print(mapping.shape)
print(mapping.head())

# ============================================================
# 5. station dictionary 로드
# ============================================================

station_dict = read_csv_safe(station_dict_path)

station_dict = station_dict.rename(columns={
    "전철역코드": "subway_station_code",
    "전철역명": "station_name",
    "호선": "dict_line",
    "외부코드": "external_code"
})

station_dict["station_name_clean"] = clean_name(station_dict["station_name"])

station_dict["STATS_STATION_ID"] = pd.to_numeric(
    station_dict["external_code"].astype(str).str.strip(),
    errors="coerce"
)

station_dict = station_dict.dropna(subset=["STATS_STATION_ID"]).copy()
station_dict["STATS_STATION_ID"] = station_dict["STATS_STATION_ID"].astype(int)

station_dict = station_dict[[
    "STATS_STATION_ID",
    "station_name",
    "station_name_clean",
    "dict_line",
    "subway_station_code",
    "external_code"
]].drop_duplicates()

print("\n[STATION DICT]")
print(station_dict.shape)
print(station_dict.head())

# ============================================================
# 6. station activity 로드 + valid dictionary 생성
# ============================================================

station_30 = pd.read_parquet(station_30min_path)
station_daily = pd.read_parquet(station_daily_path)

station_30["STATION_ID"] = pd.to_numeric(station_30["STATION_ID"], errors="coerce")
station_daily["STATION_ID"] = pd.to_numeric(station_daily["STATION_ID"], errors="coerce")

activity_ids = (
    set(station_30["STATION_ID"].dropna().astype(int).unique()) |
    set(station_daily["STATION_ID"].dropna().astype(int).unique())
)

station_dict_valid = station_dict[
    station_dict["STATS_STATION_ID"].isin(activity_ids)
].copy()

print("\n[VALID STATION DICT]")
print(station_dict_valid.shape)
print("unique STATS_STATION_ID:", station_dict_valid["STATS_STATION_ID"].nunique())

# mapping에 STATS_STATION_ID 붙이기
mapping_hc = mapping.merge(
    station_dict_valid,
    left_on="nearest_station_clean",
    right_on="station_name_clean",
    how="inner"
)

mapping_hc = (
    mapping_hc
    .sort_values(["CELL_ID", "nearest_station_distance_m", "STATS_STATION_ID"])
    .drop_duplicates(subset=["CELL_ID"])
    .copy()
)

print("\n[HIGH CONFIDENCE CELL-STATION MAPPING]")
print("mapping rows:", len(mapping))
print("hc rows:", len(mapping_hc))
print("unique STATS_STATION_ID:", mapping_hc["STATS_STATION_ID"].nunique())

# ============================================================
# 7. 핵심 수정: CELL 단위가 아니라 STATION 단위로 주변 mobility 집계
# ============================================================

cell_with_station = cell_mobility.merge(
    mapping_hc,
    on="CELL_ID",
    how="inner"
)

print("\n[CELL MOBILITY + STATION MAPPING]")
print(cell_with_station.shape)
print(cell_with_station.head())

station_surrounding_mobility = (
    cell_with_station
    .groupby(["STATS_STATION_ID", "station_name", "dict_line"], as_index=False)
    .agg(
        surrounding_cell_count=("CELL_ID", "count"),
        surrounding_total_flow=("actual_total_flow", "sum"),
        surrounding_outflow=("mobility_out_sum", "sum"),
        surrounding_inflow=("mobility_in_sum", "sum"),
        surrounding_net_flow=("mobility_net_sum", "sum"),
        surrounding_abs_net_flow=("actual_abs_net_flow", "sum"),
        mean_cell_flow=("actual_total_flow", "mean"),
        median_cell_flow=("actual_total_flow", "median"),
        mean_distance_m=("nearest_station_distance_m", "mean"),
        min_distance_m=("nearest_station_distance_m", "min"),
        max_distance_m=("nearest_station_distance_m", "max")
    )
)

station_surrounding_mobility.to_csv(
    station_mobility_output,
    index=False,
    encoding="utf-8-sig"
)

print("\n[STATION SURROUNDING MOBILITY]")
print(station_surrounding_mobility.shape)
print(station_surrounding_mobility.head())

# ============================================================
# 8. station activity 집계
# ============================================================

for c in ["od_out_cnt", "od_in_cnt", "od_net_cnt"]:
    station_30[c] = pd.to_numeric(station_30[c], errors="coerce").fillna(0)
    station_daily[c] = pd.to_numeric(station_daily[c], errors="coerce").fillna(0)

station_30_agg = (
    station_30
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_30_out_sum=("od_out_cnt", "sum"),
        station_30_in_sum=("od_in_cnt", "sum"),
        station_30_net_sum=("od_net_cnt", "sum"),
        station_30_record_count=("datetime", "count")
    )
)

station_30_agg["station_30_total_activity"] = (
    station_30_agg["station_30_out_sum"] +
    station_30_agg["station_30_in_sum"]
)

station_daily_agg = (
    station_daily
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_daily_out_sum=("od_out_cnt", "sum"),
        station_daily_in_sum=("od_in_cnt", "sum"),
        station_daily_net_sum=("od_net_cnt", "sum"),
        station_daily_record_count=("datetime", "count")
    )
)

station_daily_agg["station_daily_total_activity"] = (
    station_daily_agg["station_daily_out_sum"] +
    station_daily_agg["station_daily_in_sum"]
)

station_activity = station_30_agg.merge(
    station_daily_agg,
    on="STATS_STATION_ID",
    how="outer"
).fillna(0)

station_activity["station_total_activity"] = (
    station_activity["station_30_total_activity"] +
    station_activity["station_daily_total_activity"]
)

station_activity["station_total_in"] = (
    station_activity["station_30_in_sum"] +
    station_activity["station_daily_in_sum"]
)

station_activity["station_total_out"] = (
    station_activity["station_30_out_sum"] +
    station_activity["station_daily_out_sum"]
)

station_activity["station_total_net"] = (
    station_activity["station_total_in"] -
    station_activity["station_total_out"]
)

print("\n[STATION ACTIVITY]")
print(station_activity.shape)
print(station_activity.head())

# ============================================================
# 9. 최종: 역 주변 생활이동량 vs 지하철 activity 비교
# ============================================================

final_station = station_surrounding_mobility.merge(
    station_activity,
    on="STATS_STATION_ID",
    how="inner"
)

print("\n[FINAL STATION LEVEL MERGE]")
print(final_station.shape)
print(final_station.head())

# ============================================================
# 10. 연구 지표 생성
# ============================================================

final_station["surrounding_total_flow_norm"] = normalize_minmax(
    final_station["surrounding_total_flow"]
)

final_station["station_activity_norm"] = normalize_minmax(
    final_station["station_total_activity"]
)

final_station["distance_norm"] = normalize_minmax(
    final_station["mean_distance_m"]
)

# 핵심 1: 주변 생활이동은 큰데 지하철 activity는 낮은 역
final_station["subway_representation_gap"] = (
    final_station["surrounding_total_flow_norm"] -
    final_station["station_activity_norm"]
)

# 핵심 2: hidden demand
final_station["hidden_demand_score"] = (
    final_station["sur

read success: CELL_250x250(UTMK_epsg 5179).csv / utf-8-sig

[CELL MASTER]
(207205, 3)
      CELL_ID  CELL_X   CELL_Y
0  나사99757775  899875  1977875
1  다사18001325  918125  1913375
2  다사31254225  931375  1942375
3  다사33752875  933875  1928875
4  다바40009800  940125  1898125

[MOBILITY RAW]
(630941, 5)
    datetime CELL_ID_BASE  out_cnt  in_cnt  net_cnt
0 2023-01-02           26      7.0     0.0     -7.0
1 2023-01-02           27     10.5     0.0    -10.5
2 2023-01-02           29     21.0     0.0    -21.0
3 2023-01-02           30     17.5     7.0    -10.5
4 2023-01-02        42110     10.5     0.0    -10.5

[CELL INDEX CHECK]
valid ratio: 1.0
min: 26
max: 44825
cell master rows: 207205

[MOBILITY RESTORED]
    datetime CELL_ID_BASE  out_cnt  in_cnt  net_cnt  CELL_INDEX     CELL_ID  \
0 2023-01-02           26      7.0     0.0     -7.0          26  다사62256100   
1 2023-01-02           27     10.5     0.0    -10.5          27  다사70256000   
2 2023-01-02           29     21.0     0.0    -21

In [9]:
# ============================================================
# STATION-LEVEL RESEARCH PIPELINE TEST
# 목적:
# 지하철역 주변 cell 생활이동량 집계
# vs 역별 subway activity 비교
#
# 데이터 1개씩만 사용
# ============================================================

import os
import pandas as pd
import numpy as np

# ============================================================
# 0. 경로
# ============================================================

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\station_level_research_test"
os.makedirs(output_dir, exist_ok=True)

cell_master_path = r"C:\Users\82108\Desktop\새 폴더 (8)\CELL_250x250(UTMK_epsg 5179).csv"

mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv"

station_dict_path = r"C:\Users\82108\Desktop\새 폴더 (8)\서울교통공사_역명 지하철역 검색.csv"

mobility_path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

station_30min_path = r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output\TBDM_TRANSIT_STAT_SUBWAY_20170102_station_stats.parquet"

station_daily_path = r"C:\Users\82108\Desktop\새 폴더 (8)\od_output\od_output\202002_TBDM_TRNSIT_OD_SUBWAY_20191001_station.parquet"

station_level_output_path = os.path.join(output_dir, "STATION_LEVEL_TEST_dataset.csv")
top_gap_output_path = os.path.join(output_dir, "STATION_LEVEL_TEST_top_gap.csv")
cell_join_output_path = os.path.join(output_dir, "CELL_JOIN_TEST_debug.csv")


# ============================================================
# 1. 유틸
# ============================================================

def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")


def normalize_minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    if len(s) == 0 or s.max() == s.min():
        return pd.Series(0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())


def clean_name(s):
    return (
        s.astype(str)
        .str.replace("'", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )


# ============================================================
# 2. CELL master 로드
# ============================================================

cell_master = read_csv_safe(cell_master_path)

cell_master = cell_master[["CELL_ID", "CELL_X", "CELL_Y"]].copy()
cell_master["CELL_X"] = pd.to_numeric(cell_master["CELL_X"], errors="coerce")
cell_master["CELL_Y"] = pd.to_numeric(cell_master["CELL_Y"], errors="coerce")

print("\n[CELL MASTER]")
print(cell_master.shape)
print(cell_master.head())


# ============================================================
# 3. mobility 로드 + CELL_ID_BASE -> 실제 CELL_ID 복원
# ============================================================

mobility = pd.read_parquet(mobility_path)

print("\n[MOBILITY RAW]")
print(mobility.shape)
print(mobility.head())

mobility["CELL_INDEX"] = pd.to_numeric(
    mobility["CELL_ID_BASE"],
    errors="coerce"
)

mobility = mobility.dropna(subset=["CELL_INDEX"]).copy()
mobility["CELL_INDEX"] = mobility["CELL_INDEX"].astype(int)

valid_idx = (
    (mobility["CELL_INDEX"] >= 0) &
    (mobility["CELL_INDEX"] < len(cell_master))
)

print("\n[CELL INDEX CHECK]")
print("valid ratio:", valid_idx.mean())
print("min:", mobility["CELL_INDEX"].min())
print("max:", mobility["CELL_INDEX"].max())
print("cell master rows:", len(cell_master))

mobility = mobility[valid_idx].copy().reset_index(drop=True)

matched_cell = cell_master.iloc[mobility["CELL_INDEX"].values].reset_index(drop=True)

mobility["CELL_ID"] = matched_cell["CELL_ID"].values
mobility["CELL_X"] = matched_cell["CELL_X"].values
mobility["CELL_Y"] = matched_cell["CELL_Y"].values

for c in ["out_cnt", "in_cnt", "net_cnt"]:
    mobility[c] = pd.to_numeric(mobility[c], errors="coerce").fillna(0)

print("\n[MOBILITY RESTORED]")
print(mobility.head())


# ============================================================
# 4. CELL 단위 mobility 집계
# ============================================================

cell_mobility = (
    mobility
    .groupby("CELL_ID", as_index=False)
    .agg(
        CELL_X=("CELL_X", "first"),
        CELL_Y=("CELL_Y", "first"),
        mobility_out_sum=("out_cnt", "sum"),
        mobility_in_sum=("in_cnt", "sum"),
        mobility_net_sum=("net_cnt", "sum"),
        mobility_record_count=("datetime", "count")
    )
)

cell_mobility["actual_total_flow"] = (
    cell_mobility["mobility_out_sum"] +
    cell_mobility["mobility_in_sum"]
)

cell_mobility["actual_abs_net_flow"] = cell_mobility["mobility_net_sum"].abs()

print("\n[CELL MOBILITY]")
print(cell_mobility.shape)
print(cell_mobility.head())


# ============================================================
# 5. cell -> nearest station mapping 로드
# ============================================================

mapping = read_csv_safe(mapping_path)

mapping = mapping.rename(columns={
    "STATION_ID": "MASTER_STATION_ID"
})

mapping["nearest_station_clean"] = clean_name(mapping["nearest_station"])

print("\n[MAPPING]")
print(mapping.shape)
print(mapping.head())


# ============================================================
# 6. station dictionary 로드
#    서울교통공사_역명 지하철역 검색.csv = ID bridge
# ============================================================

station_dict = read_csv_safe(station_dict_path)

station_dict = station_dict.rename(columns={
    "전철역코드": "subway_station_code",
    "전철역명": "station_name",
    "호선": "dict_line",
    "외부코드": "external_code"
})

station_dict["station_name_clean"] = clean_name(station_dict["station_name"])

station_dict["STATS_STATION_ID"] = pd.to_numeric(
    station_dict["external_code"].astype(str).str.strip(),
    errors="coerce"
)

station_dict = station_dict.dropna(subset=["STATS_STATION_ID"]).copy()
station_dict["STATS_STATION_ID"] = station_dict["STATS_STATION_ID"].astype(int)

station_dict = station_dict[[
    "STATS_STATION_ID",
    "station_name",
    "station_name_clean",
    "dict_line",
    "subway_station_code",
    "external_code"
]].drop_duplicates()

print("\n[STATION DICT]")
print(station_dict.shape)
print(station_dict.head())


# ============================================================
# 7. station activity 로드 + station id 존재하는 dictionary만 사용
# ============================================================

station_30 = pd.read_parquet(station_30min_path)
station_daily = pd.read_parquet(station_daily_path)

station_30["STATION_ID"] = pd.to_numeric(station_30["STATION_ID"], errors="coerce")
station_daily["STATION_ID"] = pd.to_numeric(station_daily["STATION_ID"], errors="coerce")

activity_ids = (
    set(station_30["STATION_ID"].dropna().astype(int).unique())
    |
    set(station_daily["STATION_ID"].dropna().astype(int).unique())
)

station_dict_valid = station_dict[
    station_dict["STATS_STATION_ID"].isin(activity_ids)
].copy()

print("\n[VALID DICT]")
print(station_dict_valid.shape)
print("unique STATS_STATION_ID:", station_dict_valid["STATS_STATION_ID"].nunique())


# ============================================================
# 8. mapping에 STATS_STATION_ID 부여
#    unmatched는 버림
# ============================================================

mapping_hc = mapping.merge(
    station_dict_valid,
    left_on="nearest_station_clean",
    right_on="station_name_clean",
    how="inner"
)

mapping_hc = (
    mapping_hc
    .sort_values(["CELL_ID", "nearest_station_distance_m", "STATS_STATION_ID"])
    .drop_duplicates(subset=["CELL_ID"])
    .copy()
)

print("\n[HIGH CONFIDENCE MAPPING]")
print("mapping rows:", len(mapping))
print("hc rows:", len(mapping_hc))
print("kept ratio:", len(mapping_hc) / len(mapping))
print("unique stats station:", mapping_hc["STATS_STATION_ID"].nunique())


# ============================================================
# 9. 핵심: CELL mobility + mapping 결합
#    그 다음 STATION 기준으로 주변 mobility 집계
# ============================================================

cell_join = mapping_hc.merge(
    cell_mobility,
    on="CELL_ID",
    how="left",
    suffixes=("", "_mobility")
)

# mobility 없는 cell은 0으로 둠
for c in [
    "mobility_out_sum",
    "mobility_in_sum",
    "mobility_net_sum",
    "mobility_record_count",
    "actual_total_flow",
    "actual_abs_net_flow"
]:
    cell_join[c] = pd.to_numeric(cell_join[c], errors="coerce").fillna(0)

print("\n[CELL JOIN]")
print("mapping_hc cells:", mapping_hc["CELL_ID"].nunique())
print("cell_mobility cells:", cell_mobility["CELL_ID"].nunique())
print("joined rows:", len(cell_join))
print("cells with mobility > 0:", (cell_join["actual_total_flow"] > 0).sum())
print(cell_join[[
    "CELL_ID",
    "STATS_STATION_ID",
    "station_name",
    "nearest_station",
    "nearest_station_distance_m",
    "actual_total_flow"
]].head())

cell_join.to_csv(cell_join_output_path, index=False, encoding="utf-8-sig")


# station 주변 mobility aggregate
station_mobility = (
    cell_join
    .groupby(["STATS_STATION_ID", "station_name", "dict_line"], as_index=False)
    .agg(
        nearby_cell_count=("CELL_ID", "count"),
        active_cell_count=("actual_total_flow", lambda x: (x > 0).sum()),
        station_area_mobility_total=("actual_total_flow", "sum"),
        station_area_out_total=("mobility_out_sum", "sum"),
        station_area_in_total=("mobility_in_sum", "sum"),
        station_area_net_total=("mobility_net_sum", "sum"),
        mean_cell_flow=("actual_total_flow", "mean"),
        median_cell_flow=("actual_total_flow", "median"),
        max_cell_flow=("actual_total_flow", "max"),
        mean_distance_m=("nearest_station_distance_m", "mean"),
        min_distance_m=("nearest_station_distance_m", "min"),
        max_distance_m=("nearest_station_distance_m", "max")
    )
)

print("\n[STATION MOBILITY]")
print(station_mobility.shape)
print(station_mobility.head())


# ============================================================
# 10. station activity 집계
# ============================================================

for c in ["od_out_cnt", "od_in_cnt", "od_net_cnt"]:
    station_30[c] = pd.to_numeric(station_30[c], errors="coerce").fillna(0)
    station_daily[c] = pd.to_numeric(station_daily[c], errors="coerce").fillna(0)

station_30_agg = (
    station_30
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_30_out_sum=("od_out_cnt", "sum"),
        station_30_in_sum=("od_in_cnt", "sum"),
        station_30_net_sum=("od_net_cnt", "sum"),
        station_30_record_count=("datetime", "count")
    )
)

station_30_agg["station_30_total_activity"] = (
    station_30_agg["station_30_out_sum"] +
    station_30_agg["station_30_in_sum"]
)

station_daily_agg = (
    station_daily
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_daily_out_sum=("od_out_cnt", "sum"),
        station_daily_in_sum=("od_in_cnt", "sum"),
        station_daily_net_sum=("od_net_cnt", "sum"),
        station_daily_record_count=("datetime", "count")
    )
)

station_daily_agg["station_daily_total_activity"] = (
    station_daily_agg["station_daily_out_sum"] +
    station_daily_agg["station_daily_in_sum"]
)

station_activity = station_30_agg.merge(
    station_daily_agg,
    on="STATS_STATION_ID",
    how="outer"
).fillna(0)

station_activity["station_total_activity"] = (
    station_activity["station_30_total_activity"] +
    station_activity["station_daily_total_activity"]
)

station_activity["station_total_in"] = (
    station_activity["station_30_in_sum"] +
    station_activity["station_daily_in_sum"]
)

station_activity["station_total_out"] = (
    station_activity["station_30_out_sum"] +
    station_activity["station_daily_out_sum"]
)

station_activity["station_total_net"] = (
    station_activity["station_total_in"] -
    station_activity["station_total_out"]
)

print("\n[STATION ACTIVITY]")
print(station_activity.shape)
print(station_activity.head())


# ============================================================
# 11. 최종 station-level merge
# ============================================================

station_level = station_mobility.merge(
    station_activity,
    on="STATS_STATION_ID",
    how="left"
)

for c in station_level.columns:
    if c.startswith("station_") and station_level[c].dtype != "object":
        station_level[c] = station_level[c].fillna(0)

print("\n[STATION LEVEL MERGE]")
print(station_level.shape)
print(station_level.head())


# ============================================================
# 12. 연구 지표 계산
# ============================================================

station_level["mobility_norm"] = normalize_minmax(
    station_level["station_area_mobility_total"]
)

station_level["subway_activity_norm"] = normalize_minmax(
    station_level["station_total_activity"]
)

station_level["mean_distance_norm"] = normalize_minmax(
    station_level["mean_distance_m"]
)

# 실제 주변 이동량은 높은데, 지하철 activity는 낮은 역
station_level["subway_under_representation_gap"] = (
    station_level["mobility_norm"] -
    station_level["subway_activity_norm"]
)

# 지하철 activity는 높은데, 주변 생활이동량은 낮은 역
station_level["subway_over_representation_score"] = (
    station_level["subway_activity_norm"] *
    (1 - station_level["mobility_norm"])
)

# 역 주변 mobility 자체가 높은 곳
station_level["hidden_demand_score"] = (
    station_level["mobility_norm"] *
    (1 - station_level["subway_activity_norm"])
)

# 역 주변 cell은 많고 실제 이동도 큰 곳
station_level["station_area_demand_score"] = (
    station_level["mobility_norm"] *
    normalize_minmax(station_level["active_cell_count"])
)

q_hidden = station_level["hidden_demand_score"].quantile(0.90)
q_gap = station_level["subway_under_representation_gap"].quantile(0.90)
q_over = station_level["subway_over_representation_score"].quantile(0.90)

conditions = [
    station_level["hidden_demand_score"] >= q_hidden,
    station_level["subway_under_representation_gap"] >= q_gap,
    station_level["subway_over_representation_score"] >= q_over
]

choices = [
    "hidden_demand_station",
    "subway_under_represented",
    "subway_over_represented"
]

station_level["station_type"] = np.select(
    conditions,
    choices,
    default="normal"
)

top_gap = (
    station_level
    .sort_values("hidden_demand_score", ascending=False)
    .head(100)
)

# ============================================================
# 13. 저장
# ============================================================

station_level.to_csv(
    station_level_output_path,
    index=False,
    encoding="utf-8-sig"
)

top_gap.to_csv(
    top_gap_output_path,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 14. 결과 확인
# ============================================================

print("\n================================================")
print("DONE")
print("================================================")
print("station level:", station_level_output_path)
print("top gap:", top_gap_output_path)
print("cell join debug:", cell_join_output_path)

print("\n[FINAL STATION LEVEL SHAPE]")
print(station_level.shape)

print("\n[KEY METRICS]")
print(station_level[[
    "station_area_mobility_total",
    "station_total_activity",
    "mobility_norm",
    "subway_activity_norm",
    "subway_under_representation_gap",
    "subway_over_representation_score",
    "hidden_demand_score",
    "nearby_cell_count",
    "active_cell_count",
    "mean_distance_m"
]].describe())

print("\n[STATION TYPE COUNTS]")
print(station_level["station_type"].value_counts())

print("\n[TOP HIDDEN / GAP STATIONS]")
print(top_gap[[
    "STATS_STATION_ID",
    "station_name",
    "dict_line",
    "nearby_cell_count",
    "active_cell_count",
    "station_area_mobility_total",
    "station_total_activity",
    "mobility_norm",
    "subway_activity_norm",
    "hidden_demand_score",
    "subway_under_representation_gap",
    "station_type"
]].head(30))

read success: CELL_250x250(UTMK_epsg 5179).csv / utf-8-sig

[CELL MASTER]
(207205, 3)
      CELL_ID  CELL_X   CELL_Y
0  나사99757775  899875  1977875
1  다사18001325  918125  1913375
2  다사31254225  931375  1942375
3  다사33752875  933875  1928875
4  다바40009800  940125  1898125

[MOBILITY RAW]
(630941, 5)
    datetime CELL_ID_BASE  out_cnt  in_cnt  net_cnt
0 2023-01-02           26      7.0     0.0     -7.0
1 2023-01-02           27     10.5     0.0    -10.5
2 2023-01-02           29     21.0     0.0    -21.0
3 2023-01-02           30     17.5     7.0    -10.5
4 2023-01-02        42110     10.5     0.0    -10.5

[CELL INDEX CHECK]
valid ratio: 1.0
min: 26
max: 44825
cell master rows: 207205

[MOBILITY RESTORED]
    datetime CELL_ID_BASE  out_cnt  in_cnt  net_cnt  CELL_INDEX     CELL_ID  \
0 2023-01-02           26      7.0     0.0     -7.0          26  다사62256100   
1 2023-01-02           27     10.5     0.0    -10.5          27  다사70256000   
2 2023-01-02           29     21.0     0.0    -21

In [11]:
path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

df = pd.read_parquet(path)

print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
print(df.head(20))
print(df.describe(include="all"))

(630941, 5)
['datetime', 'CELL_ID_BASE', 'out_cnt', 'in_cnt', 'net_cnt']
datetime        datetime64[ns]
CELL_ID_BASE    string[python]
out_cnt                float64
in_cnt                 float64
net_cnt                float64
dtype: object
     datetime CELL_ID_BASE  out_cnt  in_cnt  net_cnt
0  2023-01-02           26      7.0     0.0     -7.0
1  2023-01-02           27     10.5     0.0    -10.5
2  2023-01-02           29     21.0     0.0    -21.0
3  2023-01-02           30     17.5     7.0    -10.5
4  2023-01-02        42110     10.5     0.0    -10.5
5  2023-01-02        42130     17.5     0.0    -17.5
6  2023-01-02        42150     21.0     3.5    -17.5
7  2023-01-02        42210     17.5     0.0    -17.5
8  2023-01-02        42720     28.0     3.5    -24.5
9  2023-01-02        42730      3.5     0.0     -3.5
10 2023-01-02        42760     10.5     0.0    -10.5
11 2023-01-02        42770      7.0     0.0     -7.0
12 2023-01-02        42820      7.0     0.0     -7.0
13 2023-01-02   

In [12]:
import os
import pandas as pd
import numpy as np

# =========================
# 0. 경로
# =========================
output_dir = r"C:\Users\82108\Desktop\sample_data_check"
os.makedirs(output_dir, exist_ok=True)

paths = {
    "cell_master": r"C:\Users\82108\Desktop\새 폴더 (8)\CELL_250x250(UTMK_epsg 5179).csv",
    "station_mapping": r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv",
    "station_dict": r"C:\Users\82108\Desktop\새 폴더 (8)\서울교통공사_역명 지하철역 검색.csv",
    "mobility_cell": r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet",
    "station_30min": r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output\TBDM_TRANSIT_STAT_SUBWAY_20170102_station_stats.parquet",
    "station_daily": r"C:\Users\82108\Desktop\새 폴더 (8)\od_output\od_output\202002_TBDM_TRNSIT_OD_SUBWAY_20191001_station.parquet",
}

# =========================
# 1. 읽기 함수
# =========================
def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")

def read_any(path):
    if path.lower().endswith(".parquet"):
        return pd.read_parquet(path)
    return read_csv_safe(path)

# =========================
# 2. 기본 구조 확인
# =========================
def inspect_df(name, df):
    print("\n" + "=" * 90)
    print(f"[DATASET] {name}")
    print("=" * 90)

    print("\n[SHAPE]")
    print(df.shape)

    print("\n[COLUMNS]")
    print(df.columns.tolist())

    print("\n[DTYPES]")
    print(df.dtypes)

    print("\n[HEAD]")
    print(df.head(10))

    print("\n[NULL COUNT]")
    print(df.isnull().sum())

    print("\n[MEMORY]")
    mem = df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"{mem:.2f} MB")

    preview_path = os.path.join(output_dir, f"{name}_preview.csv")
    df.head(1000).to_csv(preview_path, index=False, encoding="utf-8-sig")
    print("\n[SAVED PREVIEW]")
    print(preview_path)

# =========================
# 3. 하나씩 읽고 확인
# =========================
loaded = {}

for name, path in paths.items():
    print("\n\nLOADING:", name)
    df = read_any(path)
    loaded[name] = df
    inspect_df(name, df)

# =========================
# 4. mobility CELL_ID_BASE 복원 가능성 확인
# =========================
print("\n" + "=" * 90)
print("[CHECK] mobility CELL_ID_BASE -> cell_master index")
print("=" * 90)

cell_master = loaded["cell_master"].copy()
mobility = loaded["mobility_cell"].copy()

cell_master = cell_master.reset_index().rename(columns={"index": "CELL_INDEX"})

mobility["CELL_INDEX"] = pd.to_numeric(
    mobility["CELL_ID_BASE"],
    errors="coerce"
)

valid_idx = (
    mobility["CELL_INDEX"].notna()
    & (mobility["CELL_INDEX"] >= 0)
    & (mobility["CELL_INDEX"] < len(cell_master))
)

print("mobility rows:", len(mobility))
print("CELL_ID_BASE unique:", mobility["CELL_ID_BASE"].nunique())
print("CELL_INDEX min:", mobility["CELL_INDEX"].min())
print("CELL_INDEX max:", mobility["CELL_INDEX"].max())
print("cell_master rows:", len(cell_master))
print("valid index ratio:", valid_idx.mean())

sample_idx = mobility.loc[valid_idx, "CELL_INDEX"].astype(int).drop_duplicates().head(20)

print("\n[SAMPLE INDEX -> CELL_ID]")
for idx in sample_idx:
    row = cell_master.loc[cell_master["CELL_INDEX"] == idx].iloc[0]
    print(idx, "->", row["CELL_ID"], row["CELL_X"], row["CELL_Y"])

# =========================
# 5. 안전한 방식으로 CELL_ID 복원 테스트
# =========================
mobility_restore_test = mobility.copy()
mobility_restore_test = mobility_restore_test[valid_idx].copy()
mobility_restore_test["CELL_INDEX"] = mobility_restore_test["CELL_INDEX"].astype(int)

mobility_restore_test = mobility_restore_test.merge(
    cell_master[["CELL_INDEX", "CELL_ID", "CELL_X", "CELL_Y"]],
    on="CELL_INDEX",
    how="left"
)

print("\n[RESTORE TEST]")
print("restored rows:", len(mobility_restore_test))
print("restored CELL_ID unique:", mobility_restore_test["CELL_ID"].nunique())
print(mobility_restore_test.head(20))

restore_preview_path = os.path.join(output_dir, "mobility_restored_preview.csv")
mobility_restore_test.head(5000).to_csv(
    restore_preview_path,
    index=False,
    encoding="utf-8-sig"
)
print("\n[SAVED RESTORE PREVIEW]")
print(restore_preview_path)

# =========================
# 6. station id 겹침 확인
# =========================
print("\n" + "=" * 90)
print("[CHECK] station id overlap")
print("=" * 90)

station_30 = loaded["station_30min"].copy()
station_daily = loaded["station_daily"].copy()
station_dict = loaded["station_dict"].copy()

station_30_ids = set(pd.to_numeric(station_30["STATION_ID"], errors="coerce").dropna().astype(int))
station_daily_ids = set(pd.to_numeric(station_daily["STATION_ID"], errors="coerce").dropna().astype(int))

station_dict = station_dict.rename(columns={
    "전철역코드": "subway_station_code",
    "전철역명": "station_name",
    "호선": "line",
    "외부코드": "external_code"
})

station_dict["STATS_STATION_ID"] = pd.to_numeric(
    station_dict["external_code"].astype(str).str.strip(),
    errors="coerce"
)

dict_ids = set(station_dict["STATS_STATION_ID"].dropna().astype(int))

print("30min ids:", len(station_30_ids))
print("daily ids:", len(station_daily_ids))
print("dict numeric ids:", len(dict_ids))
print("30min ∩ daily:", len(station_30_ids & station_daily_ids))
print("dict ∩ 30min:", len(dict_ids & station_30_ids))
print("dict ∩ daily:", len(dict_ids & station_daily_ids))

print("\n[OVERLAP SAMPLE]")
print(sorted(list(dict_ids & station_30_ids))[:50])

# =========================
# 7. mapping과 restored mobility overlap 확인
# =========================
print("\n" + "=" * 90)
print("[CHECK] restored mobility CELL_ID ∩ station mapping CELL_ID")
print("=" * 90)

mapping = loaded["station_mapping"].copy()

mobility_cell_ids = set(mobility_restore_test["CELL_ID"].dropna().astype(str))
mapping_cell_ids = set(mapping["CELL_ID"].dropna().astype(str))

print("restored mobility unique CELL_ID:", len(mobility_cell_ids))
print("station mapping unique CELL_ID:", len(mapping_cell_ids))
print("overlap:", len(mobility_cell_ids & mapping_cell_ids))

print("\n[OVERLAP SAMPLE]")
print(list(mobility_cell_ids & mapping_cell_ids)[:30])

print("\nDONE")
print("output folder:", output_dir)



LOADING: cell_master
read success: CELL_250x250(UTMK_epsg 5179).csv / utf-8-sig

[DATASET] cell_master

[SHAPE]
(207205, 3)

[COLUMNS]
['CELL_ID', 'CELL_X', 'CELL_Y']

[DTYPES]
CELL_ID    object
CELL_X      int64
CELL_Y      int64
dtype: object

[HEAD]
      CELL_ID  CELL_X   CELL_Y
0  나사99757775  899875  1977875
1  다사18001325  918125  1913375
2  다사31254225  931375  1942375
3  다사33752875  933875  1928875
4  다바40009800  940125  1898125
5  다사53005725  953125  1957375
6  다사53255000  953375  1950125
7  다아53752450  953875  2024625
8  다아58500350  958625  2003625
9  다사59254775  959375  1947875

[NULL COUNT]
CELL_ID    0
CELL_X     0
CELL_Y     0
dtype: int64

[MEMORY]
20.16 MB

[SAVED PREVIEW]
C:\Users\82108\Desktop\sample_data_check\cell_master_preview.csv


LOADING: station_mapping
read success: cell_to_nearest_station_VALID_5km.csv / utf-8-sig

[DATASET] station_mapping

[SHAPE]
(85543, 19)

[COLUMNS]
['CELL_ID', 'CELL_X', 'CELL_Y', 'STATION_ID', 'nearest_station', 'nearest_line', 'stati

In [14]:
import os
import pandas as pd
import numpy as np

# ============================================================
# STATION-LEVEL RESEARCH PIPELINE
# CELL_ID_BASE 혼합형 복원 반영 완료
# ============================================================

# =========================
# 0. 경로
# =========================

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\station_level_research_final_test"
os.makedirs(output_dir, exist_ok=True)

cell_master_path = r"C:\Users\82108\Desktop\새 폴더 (8)\CELL_250x250(UTMK_epsg 5179).csv"
mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv"
station_dict_path = r"C:\Users\82108\Desktop\새 폴더 (8)\서울교통공사_역명 지하철역 검색.csv"

mobility_path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

station_30min_path = r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output\TBDM_TRANSIT_STAT_SUBWAY_20170102_station_stats.parquet"
station_daily_path = r"C:\Users\82108\Desktop\새 폴더 (8)\od_output\od_output\202002_TBDM_TRNSIT_OD_SUBWAY_20191001_station.parquet"

cell_mobility_output = os.path.join(output_dir, "01_cell_mobility_restored.csv")
station_mobility_output = os.path.join(output_dir, "02_station_surrounding_mobility.csv")
final_output = os.path.join(output_dir, "03_station_level_final_gap_analysis.csv")
top_hidden_output = os.path.join(output_dir, "04_top_hidden_demand_stations.csv")
debug_join_output = os.path.join(output_dir, "debug_cell_station_join.csv")


# =========================
# 1. 유틸
# =========================

def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")


def normalize_minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    if len(s) == 0 or s.max() == s.min():
        return pd.Series(0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())


def clean_name(s):
    return (
        s.astype(str)
        .str.replace("'", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )


# =========================
# 2. CELL master 로드
# =========================

cell_master = read_csv_safe(cell_master_path)

cell_master = cell_master[["CELL_ID", "CELL_X", "CELL_Y"]].copy()
cell_master["CELL_ID"] = cell_master["CELL_ID"].astype(str).str.strip()
cell_master["CELL_X"] = pd.to_numeric(cell_master["CELL_X"], errors="coerce")
cell_master["CELL_Y"] = pd.to_numeric(cell_master["CELL_Y"], errors="coerce")

# index mapping용
cell_master_index = cell_master.reset_index().rename(columns={"index": "CELL_INDEX"})

print("\n[CELL MASTER]")
print(cell_master.shape)
print(cell_master.head())


# =========================
# 3. mobility 로드 + CELL_ID_BASE 혼합형 복원
# =========================

mobility = pd.read_parquet(mobility_path)

print("\n[MOBILITY RAW]")
print(mobility.shape)
print(mobility.head())

mobility["CELL_ID_BASE_STR"] = mobility["CELL_ID_BASE"].astype(str).str.strip()
mobility["CELL_INDEX"] = pd.to_numeric(mobility["CELL_ID_BASE_STR"], errors="coerce")

# 3-1. 숫자형 part: CELL master row index로 복원
numeric_part = mobility[mobility["CELL_INDEX"].notna()].copy()
numeric_part["CELL_INDEX"] = numeric_part["CELL_INDEX"].astype(int)

valid_numeric = (
    (numeric_part["CELL_INDEX"] >= 0) &
    (numeric_part["CELL_INDEX"] < len(cell_master_index))
)

numeric_part = numeric_part[valid_numeric].copy()

numeric_part = numeric_part.merge(
    cell_master_index[["CELL_INDEX", "CELL_ID", "CELL_X", "CELL_Y"]],
    on="CELL_INDEX",
    how="left"
)

# 3-2. 문자형 part: 이미 실제 CELL_ID로 간주
string_part = mobility[mobility["CELL_INDEX"].isna()].copy()
string_part = string_part.rename(columns={"CELL_ID_BASE_STR": "CELL_ID"})
string_part["CELL_ID"] = string_part["CELL_ID"].astype(str).str.strip()

string_part = string_part.merge(
    cell_master[["CELL_ID", "CELL_X", "CELL_Y"]],
    on="CELL_ID",
    how="left"
)

# 3-3. 합치기
mobility_restored = pd.concat(
    [numeric_part, string_part],
    ignore_index=True
)

# 좌표 매칭 실패 제거
before_rows = len(mobility_restored)
mobility_restored = mobility_restored.dropna(subset=["CELL_ID", "CELL_X", "CELL_Y"]).copy()
after_rows = len(mobility_restored)

for c in ["out_cnt", "in_cnt", "net_cnt"]:
    mobility_restored[c] = pd.to_numeric(mobility_restored[c], errors="coerce").fillna(0)

print("\n[MOBILITY RESTORE CHECK]")
print("original rows:", len(mobility))
print("numeric part rows:", len(numeric_part))
print("string part rows:", len(string_part))
print("restored rows before drop:", before_rows)
print("restored rows after drop:", after_rows)
print("restored unique CELL_ID:", mobility_restored["CELL_ID"].nunique())
print("missing CELL_ID:", mobility_restored["CELL_ID"].isna().sum())
print("missing CELL_X:", mobility_restored["CELL_X"].isna().sum())
print("missing CELL_Y:", mobility_restored["CELL_Y"].isna().sum())
print(mobility_restored[["datetime", "CELL_ID_BASE", "CELL_ID", "CELL_X", "CELL_Y", "out_cnt", "in_cnt", "net_cnt"]].head())


# =========================
# 4. CELL 단위 mobility 집계
# =========================

cell_mobility = (
    mobility_restored
    .groupby("CELL_ID", as_index=False)
    .agg(
        CELL_X=("CELL_X", "first"),
        CELL_Y=("CELL_Y", "first"),
        mobility_out_sum=("out_cnt", "sum"),
        mobility_in_sum=("in_cnt", "sum"),
        mobility_net_sum=("net_cnt", "sum"),
        mobility_out_mean=("out_cnt", "mean"),
        mobility_in_mean=("in_cnt", "mean"),
        mobility_record_count=("datetime", "count")
    )
)

cell_mobility["actual_total_flow"] = (
    cell_mobility["mobility_out_sum"] + cell_mobility["mobility_in_sum"]
)

cell_mobility["actual_abs_net_flow"] = cell_mobility["mobility_net_sum"].abs()

cell_mobility.to_csv(cell_mobility_output, index=False, encoding="utf-8-sig")

print("\n[CELL MOBILITY]")
print(cell_mobility.shape)
print(cell_mobility.head())


# =========================
# 5. cell → nearest station mapping 로드
# =========================

mapping = read_csv_safe(mapping_path)

mapping = mapping.rename(columns={
    "STATION_ID": "MASTER_STATION_ID"
})

mapping["CELL_ID"] = mapping["CELL_ID"].astype(str).str.strip()
mapping["nearest_station_clean"] = clean_name(mapping["nearest_station"])

print("\n[MAPPING]")
print(mapping.shape)
print(mapping.head())


# =========================
# 6. station dictionary 로드
# =========================

station_dict = read_csv_safe(station_dict_path)

station_dict = station_dict.rename(columns={
    "전철역코드": "subway_station_code",
    "전철역명": "station_name",
    "호선": "dict_line",
    "외부코드": "external_code"
})

station_dict["station_name_clean"] = clean_name(station_dict["station_name"])

station_dict["STATS_STATION_ID"] = pd.to_numeric(
    station_dict["external_code"].astype(str).str.strip(),
    errors="coerce"
)

station_dict = station_dict.dropna(subset=["STATS_STATION_ID"]).copy()
station_dict["STATS_STATION_ID"] = station_dict["STATS_STATION_ID"].astype(int)

station_dict = station_dict[[
    "STATS_STATION_ID",
    "station_name",
    "station_name_clean",
    "dict_line",
    "subway_station_code",
    "external_code"
]].drop_duplicates()

print("\n[STATION DICT]")
print(station_dict.shape)
print(station_dict.head())


# =========================
# 7. station activity 로드 + valid dict
# =========================

station_30 = pd.read_parquet(station_30min_path)
station_daily = pd.read_parquet(station_daily_path)

station_30["STATION_ID"] = pd.to_numeric(station_30["STATION_ID"], errors="coerce")
station_daily["STATION_ID"] = pd.to_numeric(station_daily["STATION_ID"], errors="coerce")

activity_ids = (
    set(station_30["STATION_ID"].dropna().astype(int).unique())
    |
    set(station_daily["STATION_ID"].dropna().astype(int).unique())
)

station_dict_valid = station_dict[
    station_dict["STATS_STATION_ID"].isin(activity_ids)
].copy()

print("\n[VALID STATION DICT]")
print(station_dict_valid.shape)
print("unique STATS_STATION_ID:", station_dict_valid["STATS_STATION_ID"].nunique())


# =========================
# 8. mapping에 STATS_STATION_ID 부여
# =========================

mapping_hc = mapping.merge(
    station_dict_valid,
    left_on="nearest_station_clean",
    right_on="station_name_clean",
    how="inner"
)

mapping_hc = (
    mapping_hc
    .sort_values(["CELL_ID", "nearest_station_distance_m", "STATS_STATION_ID"])
    .drop_duplicates(subset=["CELL_ID"])
    .copy()
)

print("\n[HIGH CONFIDENCE CELL-STATION MAPPING]")
print("mapping rows:", len(mapping))
print("hc rows:", len(mapping_hc))
print("kept ratio:", len(mapping_hc) / len(mapping))
print("unique STATS_STATION_ID:", mapping_hc["STATS_STATION_ID"].nunique())


# =========================
# 9. CELL mobility + station mapping 결합
# =========================

cell_with_station = mapping_hc.merge(
    cell_mobility,
    on="CELL_ID",
    how="left",
    suffixes=("", "_mobility")
)

for c in [
    "mobility_out_sum",
    "mobility_in_sum",
    "mobility_net_sum",
    "mobility_out_mean",
    "mobility_in_mean",
    "mobility_record_count",
    "actual_total_flow",
    "actual_abs_net_flow"
]:
    cell_with_station[c] = pd.to_numeric(cell_with_station[c], errors="coerce").fillna(0)

cell_with_station.to_csv(debug_join_output, index=False, encoding="utf-8-sig")

print("\n[CELL WITH STATION CHECK]")
print("mapping_hc cells:", mapping_hc["CELL_ID"].nunique())
print("cell_mobility cells:", cell_mobility["CELL_ID"].nunique())
print("joined rows:", len(cell_with_station))
print("cells with mobility > 0:", (cell_with_station["actual_total_flow"] > 0).sum())
print("total surrounding mobility:", cell_with_station["actual_total_flow"].sum())
print(cell_with_station[[
    "CELL_ID",
    "STATS_STATION_ID",
    "station_name",
    "nearest_station",
    "nearest_station_distance_m",
    "actual_total_flow"
]].head())


# =========================
# 10. STATION 주변 mobility 집계
# =========================

station_surrounding_mobility = (
    cell_with_station
    .groupby(["STATS_STATION_ID", "station_name", "dict_line"], as_index=False)
    .agg(
        surrounding_cell_count=("CELL_ID", "count"),
        active_cell_count=("actual_total_flow", lambda x: (x > 0).sum()),
        surrounding_total_flow=("actual_total_flow", "sum"),
        surrounding_outflow=("mobility_out_sum", "sum"),
        surrounding_inflow=("mobility_in_sum", "sum"),
        surrounding_net_flow=("mobility_net_sum", "sum"),
        surrounding_abs_net_flow=("actual_abs_net_flow", "sum"),
        mean_cell_flow=("actual_total_flow", "mean"),
        median_cell_flow=("actual_total_flow", "median"),
        max_cell_flow=("actual_total_flow", "max"),
        mean_distance_m=("nearest_station_distance_m", "mean"),
        min_distance_m=("nearest_station_distance_m", "min"),
        max_distance_m=("nearest_station_distance_m", "max")
    )
)

station_surrounding_mobility.to_csv(
    station_mobility_output,
    index=False,
    encoding="utf-8-sig"
)

print("\n[STATION SURROUNDING MOBILITY]")
print(station_surrounding_mobility.shape)
print(station_surrounding_mobility.head())


# =========================
# 11. station activity 집계
# =========================

for c in ["od_out_cnt", "od_in_cnt", "od_net_cnt"]:
    station_30[c] = pd.to_numeric(station_30[c], errors="coerce").fillna(0)
    station_daily[c] = pd.to_numeric(station_daily[c], errors="coerce").fillna(0)

station_30_agg = (
    station_30
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_30_out_sum=("od_out_cnt", "sum"),
        station_30_in_sum=("od_in_cnt", "sum"),
        station_30_net_sum=("od_net_cnt", "sum"),
        station_30_record_count=("datetime", "count")
    )
)

station_30_agg["station_30_total_activity"] = (
    station_30_agg["station_30_out_sum"] + station_30_agg["station_30_in_sum"]
)

station_daily_agg = (
    station_daily
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_daily_out_sum=("od_out_cnt", "sum"),
        station_daily_in_sum=("od_in_cnt", "sum"),
        station_daily_net_sum=("od_net_cnt", "sum"),
        station_daily_record_count=("datetime", "count")
    )
)

station_daily_agg["station_daily_total_activity"] = (
    station_daily_agg["station_daily_out_sum"] + station_daily_agg["station_daily_in_sum"]
)

station_activity = station_30_agg.merge(
    station_daily_agg,
    on="STATS_STATION_ID",
    how="outer"
).fillna(0)

station_activity["station_total_activity"] = (
    station_activity["station_30_total_activity"] +
    station_activity["station_daily_total_activity"]
)

station_activity["station_total_in"] = (
    station_activity["station_30_in_sum"] +
    station_activity["station_daily_in_sum"]
)

station_activity["station_total_out"] = (
    station_activity["station_30_out_sum"] +
    station_activity["station_daily_out_sum"]
)

station_activity["station_total_net"] = (
    station_activity["station_total_in"] - station_activity["station_total_out"]
)

print("\n[STATION ACTIVITY]")
print(station_activity.shape)
print(station_activity.head())


# =========================
# 12. 최종 station-level merge
# =========================

final_station = station_surrounding_mobility.merge(
    station_activity,
    on="STATS_STATION_ID",
    how="inner"
)

print("\n[FINAL STATION LEVEL MERGE]")
print(final_station.shape)
print(final_station.head())


# =========================
# 13. 연구 지표 생성
# =========================

final_station["surrounding_total_flow_norm"] = normalize_minmax(
    final_station["surrounding_total_flow"]
)

final_station["station_activity_norm"] = normalize_minmax(
    final_station["station_total_activity"]
)

final_station["distance_norm"] = normalize_minmax(
    final_station["mean_distance_m"]
)

final_station["subway_representation_gap"] = (
    final_station["surrounding_total_flow_norm"] -
    final_station["station_activity_norm"]
)

final_station["hidden_demand_score"] = (
    final_station["surrounding_total_flow_norm"] *
    (1 - final_station["station_activity_norm"])
)

final_station["distance_weighted_hidden_demand"] = (
    final_station["surrounding_total_flow_norm"] *
    final_station["distance_norm"]
)

final_station["subway_overrepresentation_score"] = (
    final_station["station_activity_norm"] *
    (1 - final_station["surrounding_total_flow_norm"])
)

q_hidden = final_station["hidden_demand_score"].quantile(0.90)
q_gap = final_station["subway_representation_gap"].quantile(0.90)
q_over = final_station["subway_overrepresentation_score"].quantile(0.90)

conditions = [
    final_station["hidden_demand_score"] >= q_hidden,
    final_station["subway_representation_gap"] >= q_gap,
    final_station["subway_overrepresentation_score"] >= q_over
]

choices = [
    "hidden_demand_station",
    "subway_under_represented",
    "subway_over_represented"
]

final_station["station_type"] = np.select(
    conditions,
    choices,
    default="normal"
)

final_station = final_station.sort_values(
    "hidden_demand_score",
    ascending=False
)

top_hidden_station = final_station.head(100)

# =========================
# 14. 저장
# =========================

final_station.to_csv(
    final_output,
    index=False,
    encoding="utf-8-sig"
)

top_hidden_station.to_csv(
    top_hidden_output,
    index=False,
    encoding="utf-8-sig"
)


# =========================
# 15. 결과 확인
# =========================

print("\n================================================")
print("DONE")
print("================================================")

print("cell mobility:", cell_mobility_output)
print("station surrounding mobility:", station_mobility_output)
print("debug join:", debug_join_output)
print("final station-level result:", final_output)
print("top hidden stations:", top_hidden_output)

print("\n[FINAL SHAPE]")
print(final_station.shape)

print("\n[KEY METRICS]")
print(final_station[[
    "surrounding_cell_count",
    "active_cell_count",
    "surrounding_total_flow",
    "station_total_activity",
    "mean_distance_m",
    "surrounding_total_flow_norm",
    "station_activity_norm",
    "subway_representation_gap",
    "hidden_demand_score",
    "distance_weighted_hidden_demand",
    "subway_overrepresentation_score"
]].describe())

print("\n[STATION TYPE COUNTS]")
print(final_station["station_type"].value_counts())

print("\n[TOP HIDDEN DEMAND STATIONS]")
print(top_hidden_station[[
    "STATS_STATION_ID",
    "station_name",
    "dict_line",
    "surrounding_cell_count",
    "active_cell_count",
    "surrounding_total_flow",
    "station_total_activity",
    "mean_distance_m",
    "subway_representation_gap",
    "hidden_demand_score",
    "station_type"
]].head(30))

read success: CELL_250x250(UTMK_epsg 5179).csv / utf-8-sig

[CELL MASTER]
(207205, 3)
      CELL_ID  CELL_X   CELL_Y
0  나사99757775  899875  1977875
1  다사18001325  918125  1913375
2  다사31254225  931375  1942375
3  다사33752875  933875  1928875
4  다바40009800  940125  1898125

[MOBILITY RAW]
(630941, 5)
    datetime CELL_ID_BASE  out_cnt  in_cnt  net_cnt
0 2023-01-02           26      7.0     0.0     -7.0
1 2023-01-02           27     10.5     0.0    -10.5
2 2023-01-02           29     21.0     0.0    -21.0
3 2023-01-02           30     17.5     7.0    -10.5
4 2023-01-02        42110     10.5     0.0    -10.5

[MOBILITY RESTORE CHECK]
original rows: 630941
numeric part rows: 2067
string part rows: 628874
restored rows before drop: 630941
restored rows after drop: 630941
restored unique CELL_ID: 28475
missing CELL_ID: 0
missing CELL_X: 0
missing CELL_Y: 0
    datetime CELL_ID_BASE     CELL_ID  CELL_X   CELL_Y  out_cnt  in_cnt  \
0 2023-01-02           26  다사62256100  962375  1961125      7.0

In [15]:
import os
import pandas as pd
import numpy as np

# ============================================================
# 0. 경로
# ============================================================

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\final_station_analysis_1file"
os.makedirs(output_dir, exist_ok=True)

cell_master_path = r"C:\Users\82108\Desktop\새 폴더 (8)\CELL_250x250(UTMK_epsg 5179).csv"
station_mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv"
station_dict_path = r"C:\Users\82108\Desktop\새 폴더 (8)\서울교통공사_역명 지하철역 검색.csv"

mobility_path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

station_30min_path = r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output\TBDM_TRANSIT_STAT_SUBWAY_20170102_station_stats.parquet"
station_daily_path = r"C:\Users\82108\Desktop\새 폴더 (8)\od_output\od_output\202002_TBDM_TRNSIT_OD_SUBWAY_20191001_station.parquet"

restored_mobility_path = os.path.join(output_dir, "01_restored_mobility_cell.csv")
cell_station_join_path = os.path.join(output_dir, "02_cell_mobility_station_join.csv")
station_result_path = os.path.join(output_dir, "03_station_level_result.csv")
top_hidden_path = os.path.join(output_dir, "04_top_hidden_demand_station.csv")


# ============================================================
# 1. 유틸
# ============================================================

def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")


def clean_name(s):
    return (
        s.astype(str)
        .str.replace("'", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )


def normalize_minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    if s.max() == s.min():
        return pd.Series(0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())


# ============================================================
# 2. CELL master 로드
# ============================================================

cell_master = read_csv_safe(cell_master_path)
cell_master = cell_master[["CELL_ID", "CELL_X", "CELL_Y"]].copy()

cell_master = cell_master.reset_index().rename(columns={
    "index": "CELL_INDEX"
})

cell_master["CELL_INDEX"] = cell_master["CELL_INDEX"].astype(int)

print("\n[CELL MASTER]")
print(cell_master.shape)
print(cell_master.head())


# ============================================================
# 3. mobility 로드 + CELL_ID 복원
# ============================================================

mobility = pd.read_parquet(mobility_path)

print("\n[MOBILITY RAW]")
print(mobility.shape)
print("CELL_ID_BASE unique:", mobility["CELL_ID_BASE"].nunique())

mobility["CELL_ID_BASE_RAW"] = mobility["CELL_ID_BASE"].astype(str).str.strip()

# 숫자만 있는 값 = cell_master index
is_numeric = mobility["CELL_ID_BASE_RAW"].str.fullmatch(r"\d+")

numeric_part = mobility[is_numeric].copy()
string_part = mobility[~is_numeric].copy()

numeric_part["CELL_INDEX"] = pd.to_numeric(
    numeric_part["CELL_ID_BASE_RAW"],
    errors="coerce"
).astype(int)

numeric_part = numeric_part.merge(
    cell_master[["CELL_INDEX", "CELL_ID", "CELL_X", "CELL_Y"]],
    on="CELL_INDEX",
    how="left"
)

# 문자형은 이미 CELL_ID
string_part["CELL_ID"] = string_part["CELL_ID_BASE_RAW"]

string_part = string_part.merge(
    cell_master[["CELL_ID", "CELL_X", "CELL_Y"]],
    on="CELL_ID",
    how="left"
)

mobility_restored = pd.concat(
    [numeric_part, string_part],
    ignore_index=True
)

print("\n[RESTORE CHECK]")
print("original rows:", len(mobility))
print("numeric rows:", len(numeric_part))
print("string rows:", len(string_part))
print("restored rows:", len(mobility_restored))
print("restored CELL_ID unique:", mobility_restored["CELL_ID"].nunique())
print("missing CELL_ID:", mobility_restored["CELL_ID"].isna().sum())
print("missing CELL_X:", mobility_restored["CELL_X"].isna().sum())

for c in ["out_cnt", "in_cnt", "net_cnt"]:
    mobility_restored[c] = pd.to_numeric(
        mobility_restored[c],
        errors="coerce"
    ).fillna(0)

# CELL 단위 mobility 집계
cell_mobility = (
    mobility_restored
    .groupby("CELL_ID", as_index=False)
    .agg(
        CELL_X=("CELL_X", "first"),
        CELL_Y=("CELL_Y", "first"),
        mobility_out_sum=("out_cnt", "sum"),
        mobility_in_sum=("in_cnt", "sum"),
        mobility_net_sum=("net_cnt", "sum"),
        record_count=("datetime", "count")
    )
)

cell_mobility["actual_total_flow"] = (
    cell_mobility["mobility_out_sum"] +
    cell_mobility["mobility_in_sum"]
)

cell_mobility["actual_abs_net_flow"] = (
    cell_mobility["mobility_net_sum"].abs()
)

cell_mobility.to_csv(
    restored_mobility_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n[CELL MOBILITY]")
print(cell_mobility.shape)
print(cell_mobility.head())


# ============================================================
# 4. station mapping 로드
# ============================================================

station_mapping = read_csv_safe(station_mapping_path)

station_mapping = station_mapping.rename(columns={
    "STATION_ID": "MASTER_STATION_ID"
})

station_mapping["nearest_station_clean"] = clean_name(
    station_mapping["nearest_station"]
)

print("\n[STATION MAPPING]")
print(station_mapping.shape)


# ============================================================
# 5. station dictionary 로드
# ============================================================

station_dict = read_csv_safe(station_dict_path)

station_dict = station_dict.rename(columns={
    "전철역코드": "subway_station_code",
    "전철역명": "station_name",
    "호선": "dict_line",
    "외부코드": "external_code"
})

station_dict["station_name_clean"] = clean_name(station_dict["station_name"])

station_dict["STATS_STATION_ID"] = pd.to_numeric(
    station_dict["external_code"].astype(str).str.strip(),
    errors="coerce"
)

station_dict = station_dict.dropna(subset=["STATS_STATION_ID"]).copy()
station_dict["STATS_STATION_ID"] = station_dict["STATS_STATION_ID"].astype(int)

station_dict = station_dict[[
    "STATS_STATION_ID",
    "station_name",
    "station_name_clean",
    "dict_line",
    "subway_station_code",
    "external_code"
]].drop_duplicates()

print("\n[STATION DICT]")
print(station_dict.shape)


# ============================================================
# 6. station activity 로드 + 집계
# ============================================================

station_30 = pd.read_parquet(station_30min_path)
station_daily = pd.read_parquet(station_daily_path)

for df in [station_30, station_daily]:
    df["STATION_ID"] = pd.to_numeric(df["STATION_ID"], errors="coerce")
    for c in ["od_out_cnt", "od_in_cnt", "od_net_cnt"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

activity_ids = (
    set(station_30["STATION_ID"].dropna().astype(int).unique())
    |
    set(station_daily["STATION_ID"].dropna().astype(int).unique())
)

station_dict_valid = station_dict[
    station_dict["STATS_STATION_ID"].isin(activity_ids)
].copy()

print("\n[VALID DICT]")
print(station_dict_valid.shape)
print("unique STATS_STATION_ID:", station_dict_valid["STATS_STATION_ID"].nunique())

# mapping에 STATS_STATION_ID 부여
mapping_hc = station_mapping.merge(
    station_dict_valid,
    left_on="nearest_station_clean",
    right_on="station_name_clean",
    how="inner"
)

mapping_hc = (
    mapping_hc
    .sort_values(["CELL_ID", "nearest_station_distance_m", "STATS_STATION_ID"])
    .drop_duplicates(subset=["CELL_ID"])
    .copy()
)

print("\n[HIGH CONFIDENCE MAPPING]")
print("mapping rows:", len(station_mapping))
print("hc rows:", len(mapping_hc))
print("unique stations:", mapping_hc["STATS_STATION_ID"].nunique())


# station activity aggregate
station_30_agg = (
    station_30
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_30_out_sum=("od_out_cnt", "sum"),
        station_30_in_sum=("od_in_cnt", "sum"),
        station_30_net_sum=("od_net_cnt", "sum"),
        station_30_record_count=("datetime", "count")
    )
)

station_30_agg["station_30_total_activity"] = (
    station_30_agg["station_30_out_sum"] +
    station_30_agg["station_30_in_sum"]
)

station_daily_agg = (
    station_daily
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_daily_out_sum=("od_out_cnt", "sum"),
        station_daily_in_sum=("od_in_cnt", "sum"),
        station_daily_net_sum=("od_net_cnt", "sum"),
        station_daily_record_count=("datetime", "count")
    )
)

station_daily_agg["station_daily_total_activity"] = (
    station_daily_agg["station_daily_out_sum"] +
    station_daily_agg["station_daily_in_sum"]
)

station_activity = station_30_agg.merge(
    station_daily_agg,
    on="STATS_STATION_ID",
    how="outer"
).fillna(0)

station_activity["station_total_activity"] = (
    station_activity["station_30_total_activity"] +
    station_activity["station_daily_total_activity"]
)

station_activity["station_total_in"] = (
    station_activity["station_30_in_sum"] +
    station_activity["station_daily_in_sum"]
)

station_activity["station_total_out"] = (
    station_activity["station_30_out_sum"] +
    station_activity["station_daily_out_sum"]
)

station_activity["station_total_net"] = (
    station_activity["station_total_in"] -
    station_activity["station_total_out"]
)

print("\n[STATION ACTIVITY]")
print(station_activity.shape)


# ============================================================
# 7. CELL mobility + station mapping 결합
# ============================================================

cell_station = mapping_hc.merge(
    cell_mobility,
    on="CELL_ID",
    how="left",
    suffixes=("", "_mobility")
)

for c in [
    "mobility_out_sum",
    "mobility_in_sum",
    "mobility_net_sum",
    "record_count",
    "actual_total_flow",
    "actual_abs_net_flow"
]:
    cell_station[c] = pd.to_numeric(
        cell_station[c],
        errors="coerce"
    ).fillna(0)

print("\n[CELL-STATION JOIN]")
print("rows:", len(cell_station))
print("cells with mobility > 0:", (cell_station["actual_total_flow"] > 0).sum())
print("total mobility in matched cells:", cell_station["actual_total_flow"].sum())

cell_station.to_csv(
    cell_station_join_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 8. STATION 단위 주변 mobility 집계
# ============================================================

station_mobility = (
    cell_station
    .groupby(["STATS_STATION_ID", "station_name", "dict_line"], as_index=False)
    .agg(
        nearby_cell_count=("CELL_ID", "count"),
        active_cell_count=("actual_total_flow", lambda x: (x > 0).sum()),
        surrounding_total_flow=("actual_total_flow", "sum"),
        surrounding_outflow=("mobility_out_sum", "sum"),
        surrounding_inflow=("mobility_in_sum", "sum"),
        surrounding_net_flow=("mobility_net_sum", "sum"),
        surrounding_abs_net_flow=("actual_abs_net_flow", "sum"),
        mean_cell_flow=("actual_total_flow", "mean"),
        max_cell_flow=("actual_total_flow", "max"),
        mean_distance_m=("nearest_station_distance_m", "mean"),
        min_distance_m=("nearest_station_distance_m", "min"),
        max_distance_m=("nearest_station_distance_m", "max")
    )
)

print("\n[STATION MOBILITY]")
print(station_mobility.shape)
print(station_mobility.head())


# ============================================================
# 9. STATION mobility + subway activity 최종 결합
# ============================================================

station_level = station_mobility.merge(
    station_activity,
    on="STATS_STATION_ID",
    how="inner"
)

print("\n[STATION LEVEL]")
print(station_level.shape)


# ============================================================
# 10. 연구 지표 계산
# ============================================================

station_level["mobility_norm"] = normalize_minmax(
    station_level["surrounding_total_flow"]
)

station_level["subway_activity_norm"] = normalize_minmax(
    station_level["station_total_activity"]
)

station_level["distance_norm"] = normalize_minmax(
    station_level["mean_distance_m"]
)

station_level["subway_representation_gap"] = (
    station_level["mobility_norm"] -
    station_level["subway_activity_norm"]
)

station_level["hidden_demand_score"] = (
    station_level["mobility_norm"] *
    (1 - station_level["subway_activity_norm"])
)

station_level["distance_weighted_hidden_demand"] = (
    station_level["mobility_norm"] *
    station_level["distance_norm"]
)

station_level["subway_overrepresentation_score"] = (
    station_level["subway_activity_norm"] *
    (1 - station_level["mobility_norm"])
)

q_hidden = station_level["hidden_demand_score"].quantile(0.90)
q_gap = station_level["subway_representation_gap"].quantile(0.90)
q_over = station_level["subway_overrepresentation_score"].quantile(0.90)

conditions = [
    station_level["hidden_demand_score"] >= q_hidden,
    station_level["subway_representation_gap"] >= q_gap,
    station_level["subway_overrepresentation_score"] >= q_over
]

choices = [
    "hidden_demand_station",
    "subway_under_represented",
    "subway_over_represented"
]

station_level["station_type"] = np.select(
    conditions,
    choices,
    default="normal"
)

station_level = station_level.sort_values(
    "hidden_demand_score",
    ascending=False
)

top_hidden = station_level.head(100)


# ============================================================
# 11. 저장
# ============================================================

station_level.to_csv(
    station_result_path,
    index=False,
    encoding="utf-8-sig"
)

top_hidden.to_csv(
    top_hidden_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 12. 결과 확인
# ============================================================

print("\n================================================")
print("DONE")
print("================================================")
print("restored mobility:", restored_mobility_path)
print("cell-station join:", cell_station_join_path)
print("station result:", station_result_path)
print("top hidden:", top_hidden_path)

print("\n[FINAL SHAPE]")
print(station_level.shape)

print("\n[KEY METRICS]")
print(station_level[[
    "nearby_cell_count",
    "active_cell_count",
    "surrounding_total_flow",
    "station_total_activity",
    "mobility_norm",
    "subway_activity_norm",
    "subway_representation_gap",
    "hidden_demand_score",
    "distance_weighted_hidden_demand",
    "subway_overrepresentation_score"
]].describe())

print("\n[STATION TYPE COUNTS]")
print(station_level["station_type"].value_counts())

print("\n[TOP HIDDEN DEMAND STATIONS]")
print(top_hidden[[
    "STATS_STATION_ID",
    "station_name",
    "dict_line",
    "nearby_cell_count",
    "active_cell_count",
    "surrounding_total_flow",
    "station_total_activity",
    "mobility_norm",
    "subway_activity_norm",
    "subway_representation_gap",
    "hidden_demand_score",
    "station_type"
]].head(30))


[CELL MASTER]
(207205, 4)
   CELL_INDEX     CELL_ID  CELL_X   CELL_Y
0           0  나사99757775  899875  1977875
1           1  다사18001325  918125  1913375
2           2  다사31254225  931375  1942375
3           3  다사33752875  933875  1928875
4           4  다바40009800  940125  1898125

[MOBILITY RAW]
(630941, 5)
CELL_ID_BASE unique: 28486

[RESTORE CHECK]
original rows: 630941
numeric rows: 2067
string rows: 628874
restored rows: 630941
restored CELL_ID unique: 28475
missing CELL_ID: 0
missing CELL_X: 0

[CELL MOBILITY]
(28475, 9)
      CELL_ID  CELL_X   CELL_Y  mobility_out_sum  mobility_in_sum  \
0  가사47509825  747625  1998375               3.5              3.5   
1  가사47759700  747875  1997125               7.0              7.0   
2  가사47759875  747875  1998875               7.0              7.0   
3  가사48759800  748875  1998125              31.5             21.0   
4  가사50009525  750125  1995375              35.0             45.5   

   mobility_net_sum  record_count  actual_total_f

In [34]:
import os
import glob
import pandas as pd
from huggingface_hub import HfApi, create_repo

# =========================================================
# 0. 설정
# =========================================================

INPUT_DIR = r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output"
OUTPUT_DIR = r"C:\Users\82108\Desktop\hf_ready_30min_od_daily"

REPO_ID = "youngbongbong/30_od_subway"
REPO_TYPE = "dataset"

PATH_IN_REPO = "daily_30min_od"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# 1. parquet 파일 찾기
# =========================================================

files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.parquet")))

print("total files:", len(files))

if len(files) == 0:
    raise FileNotFoundError(INPUT_DIR)

print("\nsample files:")
for f in files[:10]:
    print(" -", os.path.basename(f))

# =========================================================
# 2. 하루 단위 parquet 그대로 저장
# =========================================================

saved = 0

for f in files:

    try:
        df = pd.read_parquet(f)

        filename = os.path.basename(f)

        save_path = os.path.join(
            OUTPUT_DIR,
            filename
        )

        # 압축만 다시 적용
        df.to_parquet(
            save_path,
            compression="zstd",
            index=False
        )

        print("saved:", filename, df.shape)
        saved += 1

    except Exception as e:
        print("[SKIP]", os.path.basename(f))
        print(e)

print("\nSaved files:", saved)

# =========================================================
# 3. README 생성
# =========================================================

readme = """---
license: other
task_categories:
- time-series-forecasting
- feature-extraction
tags:
- urban-mobility
- transportation
- subway
- od
- seoul
- spatio-temporal
- 30min
pretty_name: Seoul Subway 30-Minute OD Data
---

# Seoul Subway 30-Minute OD Data

This repository contains daily parquet files of 30-minute subway OD-related records.

## Notes

- Original parquet structure and column names are preserved.
- Files are stored on a daily basis.
- Compression is re-applied using ZSTD.
- Intended for non-commercial research and educational use.
"""

with open(os.path.join(OUTPUT_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(readme)

# =========================================================
# 4. Hugging Face 업로드
# =========================================================

api = HfApi()

create_repo(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    exist_ok=True
)

api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    path_in_repo=PATH_IN_REPO,
    commit_message="Upload daily 30-minute OD parquet files"
)

print("\nDONE")
print("uploaded to:")
print(f"https://huggingface.co/datasets/{REPO_ID}")

total files: 3146

sample files:
 - TBDM_TRANSIT_STAT_SUBWAY_20170101_station_stats.parquet
 - TBDM_TRANSIT_STAT_SUBWAY_20170102_station_stats.parquet
 - TBDM_TRANSIT_STAT_SUBWAY_20170103_station_stats.parquet
 - TBDM_TRANSIT_STAT_SUBWAY_20170104_station_stats.parquet
 - TBDM_TRANSIT_STAT_SUBWAY_20170105_station_stats.parquet
 - TBDM_TRANSIT_STAT_SUBWAY_20170106_station_stats.parquet
 - TBDM_TRANSIT_STAT_SUBWAY_20170107_station_stats.parquet
 - TBDM_TRANSIT_STAT_SUBWAY_20170108_station_stats.parquet
 - TBDM_TRANSIT_STAT_SUBWAY_20170109_station_stats.parquet
 - TBDM_TRANSIT_STAT_SUBWAY_20170110_station_stats.parquet
saved: TBDM_TRANSIT_STAT_SUBWAY_20170101_station_stats.parquet (11841, 5)
saved: TBDM_TRANSIT_STAT_SUBWAY_20170102_station_stats.parquet (12380, 5)
saved: TBDM_TRANSIT_STAT_SUBWAY_20170103_station_stats.parquet (12381, 5)
saved: TBDM_TRANSIT_STAT_SUBWAY_20170104_station_stats.parquet (12374, 5)
saved: TBDM_TRANSIT_STAT_SUBWAY_20170105_station_stats.parquet (12397, 5)
saved: 

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


DONE
uploaded to:
https://huggingface.co/datasets/youngbongbong/30_od_subway


In [1]:
import os
import glob
import math
import shutil
import pandas as pd
from huggingface_hub import HfApi, create_repo

# =========================================================
# 0. 설정
# =========================================================

INPUT_DIR = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output"
BASE_OUTPUT_DIR = r"C:\Users\82108\Desktop\hf_ready_mobility_chunks"

REPO_ID = "youngbongbong/mobility_dataset"
REPO_TYPE = "dataset"
PATH_IN_REPO = "daily_mobility"

N_CHUNKS = 4

os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

# =========================================================
# 1. parquet 파일 찾기
# =========================================================

files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.parquet")))

print("total files:", len(files))

if len(files) == 0:
    raise FileNotFoundError(INPUT_DIR)

chunk_size = math.ceil(len(files) / N_CHUNKS)

chunks = [
    files[i:i + chunk_size]
    for i in range(0, len(files), chunk_size)
]

print("num chunks:", len(chunks))
for i, ch in enumerate(chunks, start=1):
    print(f"chunk {i}: {len(ch)} files")

# =========================================================
# 2. Hugging Face repo 준비
# =========================================================

api = HfApi()

create_repo(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    exist_ok=True
)

# =========================================================
# 3. README 업로드
# =========================================================

readme_dir = os.path.join(BASE_OUTPUT_DIR, "_readme")
os.makedirs(readme_dir, exist_ok=True)

readme = """---
license: other
task_categories:
- time-series-forecasting
- feature-extraction
tags:
- mobility
- urban-mobility
- geospatial
- transportation
- spatio-temporal
- cell-based
- seoul
pretty_name: Seoul Mobility Cell Dataset
---

# Seoul Mobility Cell Dataset

This repository contains daily cell-level mobility parquet files.

## Notes

- Original parquet structure and column names are preserved.
- Files are stored on a daily basis.
- Compression is re-applied using ZSTD.
- Intended for non-commercial research and educational use.

## Possible Use Cases

- urban mobility analysis
- hidden demand estimation
- transportation demand modeling
- spatio-temporal analysis
- mobility forecasting
- subway accessibility analysis
"""

with open(os.path.join(readme_dir, "README.md"), "w", encoding="utf-8") as f:
    f.write(readme)

api.upload_file(
    path_or_fileobj=os.path.join(readme_dir, "README.md"),
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    commit_message="Add dataset card"
)

# =========================================================
# 4. chunk 단위로 재저장 후 업로드
# =========================================================

total_saved = 0
total_skipped = 0

for chunk_idx, chunk_files in enumerate(chunks, start=1):

    print("\n" + "=" * 80)
    print(f"[CHUNK {chunk_idx}/{len(chunks)}]")
    print("=" * 80)

    chunk_output_dir = os.path.join(BASE_OUTPUT_DIR, f"chunk_{chunk_idx:02d}")

    if os.path.exists(chunk_output_dir):
        shutil.rmtree(chunk_output_dir)

    os.makedirs(chunk_output_dir, exist_ok=True)

    saved = 0
    skipped = 0

    for f in chunk_files:
        filename = os.path.basename(f)
        save_path = os.path.join(chunk_output_dir, filename)

        try:
            df = pd.read_parquet(f)

            df.to_parquet(
                save_path,
                compression="zstd",
                index=False
            )

            print("saved:", filename, df.shape)
            saved += 1

        except Exception as e:
            print("[SKIP]", filename)
            print("reason:", e)
            skipped += 1

    print(f"\n[CHUNK {chunk_idx}] saved:", saved)
    print(f"[CHUNK {chunk_idx}] skipped:", skipped)

    if saved == 0:
        print(f"[CHUNK {chunk_idx}] no valid files. upload skipped.")
        total_skipped += skipped
        continue

    # chunk 업로드
    api.upload_folder(
        folder_path=chunk_output_dir,
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        path_in_repo=PATH_IN_REPO,
        commit_message=f"Upload daily mobility parquet files chunk {chunk_idx}"
    )

    print(f"[CHUNK {chunk_idx}] uploaded.")

    total_saved += saved
    total_skipped += skipped

    # 업로드 끝난 chunk 로컬 임시파일 삭제하고 싶으면 주석 해제
    # shutil.rmtree(chunk_output_dir)

# =========================================================
# 5. 완료
# =========================================================

print("\nDONE")
print("total saved:", total_saved)
print("total skipped:", total_skipped)
print("uploaded to:")
print(f"https://huggingface.co/datasets/{REPO_ID}")

total files: 2099
num chunks: 4
chunk 1: 525 files
chunk 2: 525 files
chunk 3: 525 files
chunk 4: 524 files

[CHUNK 1/4]
saved: 202301__202301__PURPOSE_forn_250M_20230101_cell.parquet (589533, 5)
saved: 202301__202301__PURPOSE_forn_250M_20230102_cell.parquet (630941, 5)
saved: 202301__202301__PURPOSE_forn_250M_20230103_cell.parquet (637026, 5)
saved: 202301__202301__PURPOSE_forn_250M_20230104_cell.parquet (638559, 5)
saved: 202301__202301__PURPOSE_forn_250M_20230105_cell.parquet (639536, 5)
saved: 202301__202301__PURPOSE_forn_250M_20230106_cell.parquet (636782, 5)
saved: 202301__202301__PURPOSE_forn_250M_20230107_cell.parquet (609504, 5)
saved: 202301__202301__PURPOSE_forn_250M_20230108_cell.parquet (584220, 5)
saved: 202301__202301__PURPOSE_forn_250M_20230109_cell.parquet (634560, 5)
saved: 202301__202301__PURPOSE_forn_250M_20230110_cell.parquet (639595, 5)
saved: 202301__202301__PURPOSE_forn_250M_20230111_cell.parquet (640082, 5)
saved: 202301__202301__PURPOSE_forn_250M_20230112_cell

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


saved: 202309__202309__PURPOSE_in_250M_20230910_cell.parquet (890045, 5)

[CHUNK 1] saved: 524
[CHUNK 1] skipped: 1


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


saved: PURPOSE_forn_250M_20260228__PURPOSE_forn_250M_20260228_cell.parquet (653022, 5)

[CHUNK 4] saved: 524
[CHUNK 4] skipped: 0


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[CHUNK 4] uploaded.

DONE
total saved: 2098
total skipped: 1
uploaded to:
https://huggingface.co/datasets/youngbongbong/mobility_dataset


In [5]:
# ============================================================
# LIGHT ALL-IN-ONE MAP VISUALIZATION
# for station-level result: (327, 47)
# excludes cluster summary files automatically
# output: interactive HTML + CSV
# ============================================================

import os
import glob
import webbrowser
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

# ============================================================
# 0. PATH
# ============================================================

BASE_CANDIDATES = [
    r"C:\Users\82108\Desktop\새 폴더 (8)\diverse_mobility_subway_analysis_filtered",
    r"C:\Users\82108\Desktop\새 폴더 (8)\advanced_station_analysis",
]

MAPPING_PATH = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km_MASTER_ID_FILTERED.csv"

OUT_DIR = r"C:\Users\82108\Desktop\새 폴더 (8)\station_map_visual_output"
HTML_DIR = os.path.join(OUT_DIR, "html")
CSV_DIR = os.path.join(OUT_DIR, "csv")

os.makedirs(HTML_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)


# ============================================================
# 1. UTILS
# ============================================================

def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )
    return df


def read_csv_safe(path, nrows=None):
    try:
        df = pd.read_csv(path, encoding="utf-8-sig", nrows=nrows)
    except:
        df = pd.read_csv(path, encoding="cp949", nrows=nrows)
    return clean_columns(df)


def find_station_level_file():
    required_cols = {
        "MASTER_STATION_ID",
        "nearest_station",
        "nearest_line",
        "hidden_demand_score",
        "station_type",
    }

    bad_keywords = [
        "summary",
        "cluster_summary",
        "robust_hidden",
        "top",
        "cell",
        "mapping",
    ]

    candidates = []

    for base in BASE_CANDIDATES:
        if not os.path.exists(base):
            continue

        for path in glob.glob(os.path.join(base, "*.csv")):
            name = os.path.basename(path).lower()

            if any(k in name for k in bad_keywords):
                continue

            try:
                tmp = read_csv_safe(path, nrows=5)
                cols = set(tmp.columns)

                if required_cols.issubset(cols):
                    full = read_csv_safe(path)
                    score = 0

                    if full.shape[0] == 327:
                        score += 100
                    if full.shape[1] >= 40:
                        score += 50
                    if "subway_total" in name:
                        score += 30
                    if "compare" in name:
                        score += 20
                    if "advanced" in name:
                        score += 10

                    candidates.append((score, path, full.shape))

            except Exception as e:
                print("[SKIP]", path, e)

    if not candidates:
        raise FileNotFoundError("327행 역별 분석 CSV를 못 찾음. 폴더 안 파일명을 확인해야 함.")

    candidates = sorted(candidates, key=lambda x: x[0], reverse=True)

    print("\n[CANDIDATE FILES]")
    for s, p, shape in candidates:
        print(s, shape, p)

    return candidates[0][1]


# ============================================================
# 2. LOAD STATION-LEVEL ANALYSIS FILE
# ============================================================

INPUT_PATH = find_station_level_file()

print("\n[SELECTED INPUT]")
print(INPUT_PATH)

df = read_csv_safe(INPUT_PATH)

print("\n[DF SHAPE]", df.shape)
print("[DF COLS]", df.columns.tolist())

if df.shape[0] != 327:
    print("[WARN] 327행이 아님. 현재 shape:", df.shape)

required = [
    "MASTER_STATION_ID",
    "nearest_station",
    "nearest_line",
    "hidden_demand_score",
    "station_type",
]

missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f"필수 컬럼 없음: {missing_required}")


# ============================================================
# 3. ADD CLUSTER IF MISSING
# ============================================================
# 네 결과에는 cluster가 있었는데, 47개 컬럼 목록에는 cluster가 없을 수 있음.
# cluster 컬럼 없으면 01_cluster_assignment 같은 파일을 찾거나,
# 없으면 -1로 둠.

if "cluster" not in df.columns:
    print("[INFO] cluster 컬럼 없음. cluster=-1로 설정")
    df["cluster"] = -1


# ============================================================
# 4. MERGE COORDINATES FROM MAPPING FILE
# ============================================================

if {"lon", "lat"}.issubset(df.columns):
    print("[COORD] lon/lat already exists")

elif {"station_lon", "station_lat"}.issubset(df.columns):
    print("[COORD] station_lon/station_lat already exists")
    df["lon"] = df["station_lon"]
    df["lat"] = df["station_lat"]

else:
    print("\n[READ MAPPING]")
    mp = read_csv_safe(MAPPING_PATH)

    print("[MAPPING SHAPE]", mp.shape)
    print("[MAPPING COLS]", mp.columns.tolist())

    if not {"MASTER_STATION_ID", "station_lat", "station_lon"}.issubset(mp.columns):
        raise ValueError("mapping 파일에 MASTER_STATION_ID, station_lat, station_lon 필요")

    coord = (
        mp.groupby("MASTER_STATION_ID", as_index=False)
        [["station_lat", "station_lon"]]
        .mean()
    )

    df = df.merge(coord, on="MASTER_STATION_ID", how="left")

    df["lat"] = df["station_lat"]
    df["lon"] = df["station_lon"]

missing_coord = df["lat"].isna().sum()
print("\n[MISSING COORD]", missing_coord, "/", len(df))

if missing_coord > 0:
    print(df.loc[df["lat"].isna(), ["MASTER_STATION_ID", "nearest_station", "nearest_line"]].head(20))

df = df.dropna(subset=["lat", "lon"]).copy()


# ============================================================
# 5. BASIC CLEANING
# ============================================================

num_cols = [
    "mapped_cell_count",
    "active_cell_count",
    "surrounding_total_flow",
    "surrounding_outflow",
    "surrounding_inflow",
    "surrounding_netflow",
    "mean_cell_flow",
    "mean_distance_m",
    "mean_directional_imbalance",
    "morning_peak_flow",
    "daytime_flow",
    "evening_peak_flow",
    "night_flow",
    "mean_temporal_entropy",
    "active_cell_ratio",
    "station_directional_imbalance",
    "period_entropy",
    "distance_band_entropy",
    "outer_1km_plus_share",
    "subway_total_activity",
    "subway_directional_imbalance",
    "mobility_norm",
    "subway_norm",
    "mobility_minus_subway_gap",
    "hidden_demand_score",
    "subway_overrepresentation_score",
    "distance_norm",
    "distance_weighted_hidden_demand",
    "outer_hidden_demand_score",
    "diverse_living_area_score",
    "directional_mismatch_score",
]

for c in num_cols:
    if c not in df.columns:
        df[c] = 0
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["cluster"] = df["cluster"].fillna(-1).astype(int)
df["cluster_str"] = df["cluster"].astype(str)
df["station_type"] = df["station_type"].fillna("unknown").astype(str)
df["nearest_line"] = df["nearest_line"].fillna("unknown").astype(str)
df["nearest_station"] = df["nearest_station"].fillna("unknown").astype(str)

df["station_label"] = (
    df["nearest_station"].astype(str)
    + " / "
    + df["nearest_line"].astype(str)
)

# plotly size가 0이면 안 보이는 경우가 있어서 보정
df["hidden_size"] = df["hidden_demand_score"].clip(lower=0) + 0.01
df["mismatch_size"] = df["directional_mismatch_score"].clip(lower=0) + 0.005
df["outer_size"] = df["outer_hidden_demand_score"].clip(lower=0) + 0.005
df["diverse_size"] = df["diverse_living_area_score"].clip(lower=0) + 0.005
df["gap_abs_size"] = df["mobility_minus_subway_gap"].abs() + 0.005

df.to_csv(
    os.path.join(CSV_DIR, "station_level_map_ready_327.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("\n[READY DF]", df.shape)


# ============================================================
# 6. PLOTLY INTERACTIVE MAPS
# ============================================================

html_paths = []

try:
    import plotly.express as px
    import plotly.graph_objects as go

    def save_plotly(fig, filename):
        path = os.path.join(HTML_DIR, filename)
        fig.write_html(path)
        html_paths.append(path)
        print("[PLOTLY]", path)

    hover_cols = {
        "MASTER_STATION_ID": True,
        "nearest_line": True,
        "station_type": True,
        "cluster": True,
        "hidden_demand_score": ":.3f",
        "mobility_minus_subway_gap": ":.3f",
        "directional_mismatch_score": ":.3f",
        "outer_hidden_demand_score": ":.3f",
        "diverse_living_area_score": ":.3f",
        "subway_total_activity": ":,.0f",
        "surrounding_total_flow": ":,.0f",
        "lat": False,
        "lon": False,
    }

    # 01 Hidden demand
    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        size="hidden_size",
        color="hidden_demand_score",
        hover_name="nearest_station",
        hover_data=hover_cols,
        zoom=10,
        height=850,
        title="Hidden Demand Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "01_hidden_demand_map.html")

    # 02 Mobility-subway divergence
    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        size="gap_abs_size",
        color="mobility_minus_subway_gap",
        hover_name="nearest_station",
        hover_data=hover_cols,
        zoom=10,
        height=850,
        title="Mobility-Subway Divergence Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "02_mobility_subway_divergence_map.html")

    # 03 Directional mismatch
    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        size="mismatch_size",
        color="directional_mismatch_score",
        hover_name="nearest_station",
        hover_data=hover_cols,
        zoom=10,
        height=850,
        title="Directional Mismatch Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "03_directional_mismatch_map.html")

    # 04 Station type
    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        size="hidden_size",
        color="station_type",
        hover_name="nearest_station",
        hover_data=hover_cols,
        zoom=10,
        height=850,
        title="Station Type Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "04_station_type_map.html")

    # 05 Cluster
    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        size="hidden_size",
        color="cluster_str",
        hover_name="nearest_station",
        hover_data=hover_cols,
        zoom=10,
        height=850,
        title="Cluster Topology Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "05_cluster_topology_map.html")

    # 06 Outer hidden demand
    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        size="outer_size",
        color="outer_hidden_demand_score",
        hover_name="nearest_station",
        hover_data=hover_cols,
        zoom=10,
        height=850,
        title="Outer Hidden Demand Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "06_outer_hidden_demand_map.html")

    # 07 Diverse living area
    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        size="diverse_size",
        color="diverse_living_area_score",
        hover_name="nearest_station",
        hover_data=hover_cols,
        zoom=10,
        height=850,
        title="Diverse Living Area Hidden Demand Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "07_diverse_living_area_map.html")

    # 08 Temporal entropy
    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        size=df["mean_temporal_entropy"].clip(lower=0) + 0.01,
        color="mean_temporal_entropy",
        hover_name="nearest_station",
        hover_data=hover_cols,
        zoom=10,
        height=850,
        title="Temporal Entropy Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "08_temporal_entropy_map.html")

    # 09 Distance-band entropy
    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        size=df["distance_band_entropy"].clip(lower=0) + 0.01,
        color="distance_band_entropy",
        hover_name="nearest_station",
        hover_data=hover_cols,
        zoom=10,
        height=850,
        title="Distance-band Entropy Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "09_distance_band_entropy_map.html")

    # 10 Quadrant map
    hq = df["hidden_demand_score"].quantile(0.75)
    mq = df["directional_mismatch_score"].quantile(0.75)

    df["hidden_mismatch_quadrant"] = "balanced"
    df.loc[
        (df["hidden_demand_score"] >= hq)
        & (df["directional_mismatch_score"] < mq),
        "hidden_mismatch_quadrant"
    ] = "hidden only"

    df.loc[
        (df["hidden_demand_score"] < hq)
        & (df["directional_mismatch_score"] >= mq),
        "hidden_mismatch_quadrant"
    ] = "mismatch only"

    df.loc[
        (df["hidden_demand_score"] >= hq)
        & (df["directional_mismatch_score"] >= mq),
        "hidden_mismatch_quadrant"
    ] = "hidden + mismatch"

    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        size="hidden_size",
        color="hidden_mismatch_quadrant",
        hover_name="nearest_station",
        hover_data=hover_cols,
        zoom=10,
        height=850,
        title="Hidden Demand × Directional Mismatch Quadrant Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "10_hidden_mismatch_quadrant_map.html")

    # 11 Top hidden only
    top_hidden = df.sort_values("hidden_demand_score", ascending=False).head(32).copy()
    fig = px.scatter_map(
        top_hidden,
        lat="lat",
        lon="lon",
        size="hidden_size",
        color="hidden_demand_score",
        text="nearest_station",
        hover_name="nearest_station",
        hover_data=hover_cols,
        zoom=10,
        height=850,
        title="Top 32 Robust Hidden Demand Stations"
    )
    fig.update_traces(textposition="top center")
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "11_top32_robust_hidden_stations.html")

    # 12 Top mismatch only
    top_mismatch = df.sort_values("directional_mismatch_score", ascending=False).head(25).copy()
    fig = px.scatter_map(
        top_mismatch,
        lat="lat",
        lon="lon",
        size="mismatch_size",
        color="directional_mismatch_score",
        text="nearest_station",
        hover_name="nearest_station",
        hover_data=hover_cols,
        zoom=10,
        height=850,
        title="Top 25 Directional Mismatch Stations"
    )
    fig.update_traces(textposition="top center")
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "12_top25_directional_mismatch_stations.html")

except Exception as e:
    print("[SKIP PLOTLY]", e)


# ============================================================
# 7. FOLIUM MAPS
# ============================================================

try:
    import folium
    from folium.plugins import HeatMap, MarkerCluster

    center = [float(df["lat"].mean()), float(df["lon"].mean())]

    def save_folium(m, filename):
        path = os.path.join(HTML_DIR, filename)
        m.save(path)
        html_paths.append(path)
        print("[FOLIUM]", path)

    # 13 Satellite hidden demand
    m = folium.Map(location=center, zoom_start=11, tiles=None)

    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Imagery",
        name="Satellite",
        overlay=False,
        control=True
    ).add_to(m)

    folium.TileLayer("CartoDB dark_matter", name="Dark").add_to(m)
    folium.TileLayer("CartoDB positron", name="Light").add_to(m)

    for _, r in df.iterrows():
        v = r["hidden_demand_score"]

        if v >= 0.60:
            color = "red"
        elif v >= 0.45:
            color = "orange"
        elif v >= 0.30:
            color = "yellow"
        else:
            color = "blue"

        folium.CircleMarker(
            location=[r["lat"], r["lon"]],
            radius=4 + v * 16,
            color=color,
            fill=True,
            fill_opacity=0.65,
            popup=f"""
            <b>{r['nearest_station']}</b><br>
            Line: {r['nearest_line']}<br>
            Type: {r['station_type']}<br>
            Cluster: {r['cluster']}<br>
            Hidden: {r['hidden_demand_score']:.3f}<br>
            Gap: {r['mobility_minus_subway_gap']:.3f}<br>
            Mismatch: {r['directional_mismatch_score']:.3f}
            """
        ).add_to(m)

    folium.LayerControl().add_to(m)
    save_folium(m, "13_folium_satellite_hidden_demand.html")

    # 14 Hidden heatmap
    m = folium.Map(location=center, zoom_start=11, tiles="CartoDB dark_matter")
    heat_data = df[["lat", "lon", "hidden_demand_score"]].values.tolist()
    HeatMap(heat_data, radius=25, blur=18, min_opacity=0.25).add_to(m)
    save_folium(m, "14_folium_hidden_demand_heatmap.html")

    # 15 Directional mismatch satellite
    m = folium.Map(location=center, zoom_start=11, tiles=None)

    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Imagery",
        name="Satellite",
        overlay=False,
        control=True
    ).add_to(m)

    for _, r in df.iterrows():
        v = r["directional_mismatch_score"]

        folium.CircleMarker(
            location=[r["lat"], r["lon"]],
            radius=3 + v * 45,
            color="cyan",
            fill=True,
            fill_opacity=0.65,
            popup=f"""
            <b>{r['nearest_station']}</b><br>
            Line: {r['nearest_line']}<br>
            Mismatch: {r['directional_mismatch_score']:.3f}<br>
            Hidden: {r['hidden_demand_score']:.3f}
            """
        ).add_to(m)

    save_folium(m, "15_folium_satellite_directional_mismatch.html")

    # 16 Marker cluster
    m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")
    mc = MarkerCluster().add_to(m)

    for _, r in df.iterrows():
        folium.Marker(
            location=[r["lat"], r["lon"]],
            popup=f"""
            <b>{r['nearest_station']}</b><br>
            Line: {r['nearest_line']}<br>
            Type: {r['station_type']}<br>
            Hidden: {r['hidden_demand_score']:.3f}
            """
        ).add_to(mc)

    save_folium(m, "16_folium_marker_cluster.html")

    # 17 Robust hidden labeled satellite
    top = df.sort_values("hidden_demand_score", ascending=False).head(32)

    m = folium.Map(location=center, zoom_start=11, tiles=None)

    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Imagery",
        name="Satellite",
        overlay=False,
        control=True
    ).add_to(m)

    for _, r in top.iterrows():
        folium.CircleMarker(
            location=[r["lat"], r["lon"]],
            radius=7 + r["hidden_demand_score"] * 18,
            color="red",
            fill=True,
            fill_opacity=0.75,
            popup=f"""
            <b>{r['nearest_station']}</b><br>
            Line: {r['nearest_line']}<br>
            Hidden: {r['hidden_demand_score']:.3f}<br>
            Outer: {r['outer_hidden_demand_score']:.3f}<br>
            Diverse: {r['diverse_living_area_score']:.3f}<br>
            Mismatch: {r['directional_mismatch_score']:.3f}
            """
        ).add_to(m)

        folium.Marker(
            location=[r["lat"], r["lon"]],
            icon=folium.DivIcon(
                html=f"""
                <div style="
                    font-size:11px;
                    color:white;
                    background:rgba(0,0,0,0.60);
                    padding:2px 4px;
                    border-radius:4px;">
                    {r['nearest_station']}
                </div>
                """
            )
        ).add_to(m)

    save_folium(m, "17_folium_top32_robust_hidden_labeled_satellite.html")

except Exception as e:
    print("[SKIP FOLIUM]", e)


# ============================================================
# 8. PYDECK MAPS
# ============================================================

try:
    import pydeck as pdk

    def rgba_hidden(v):
        if v >= 0.60:
            return [255, 0, 0, 230]
        elif v >= 0.45:
            return [255, 100, 0, 220]
        elif v >= 0.30:
            return [255, 220, 0, 200]
        else:
            return [80, 160, 255, 130]

    def rgba_mismatch(v):
        if v >= 0.15:
            return [0, 255, 255, 235]
        elif v >= 0.08:
            return [0, 180, 255, 220]
        elif v >= 0.04:
            return [80, 120, 255, 190]
        else:
            return [180, 180, 180, 90]

    station_type_palette = {
        "normal": [170, 170, 170, 120],
        "directional_mismatch": [0, 255, 255, 235],
        "subway_high_mobility_low": [70, 130, 255, 220],
        "diverse_living_area_hidden": [255, 80, 0, 235],
        "outer_hidden_demand": [255, 0, 180, 235],
        "mobility_high_subway_low": [255, 220, 0, 235],
        "unknown": [180, 180, 180, 150],
    }

    cluster_palette = {
        0: [230, 25, 75, 220],
        1: [60, 180, 75, 220],
        2: [0, 130, 200, 220],
        3: [245, 130, 48, 220],
        4: [145, 30, 180, 220],
        5: [70, 240, 240, 220],
        -1: [180, 180, 180, 180],
    }

    df["hidden_color"] = df["hidden_demand_score"].apply(rgba_hidden)
    df["mismatch_color"] = df["directional_mismatch_score"].apply(rgba_mismatch)
    df["station_type_color"] = df["station_type"].apply(
        lambda x: station_type_palette.get(str(x), [180, 180, 180, 150])
    )
    df["cluster_color"] = df["cluster"].apply(
        lambda x: cluster_palette.get(int(x), [180, 180, 180, 180])
    )

    tooltip = {
        "html": """
        <b>{nearest_station}</b><br/>
        Line: {nearest_line}<br/>
        Type: {station_type}<br/>
        Cluster: {cluster}<br/>
        Hidden: {hidden_demand_score}<br/>
        Gap: {mobility_minus_subway_gap}<br/>
        Mismatch: {directional_mismatch_score}<br/>
        Outer: {outer_hidden_demand_score}<br/>
        Diverse: {diverse_living_area_score}
        """,
        "style": {"backgroundColor": "black", "color": "white"}
    }

    def save_deck(layers, filename, pitch=50):
        deck = pdk.Deck(
            layers=layers,
            initial_view_state=pdk.ViewState(
                latitude=float(df["lat"].mean()),
                longitude=float(df["lon"].mean()),
                zoom=10.2,
                pitch=pitch,
                bearing=0
            ),
            map_style=pdk.map_styles.CARTO_DARK,
            tooltip=tooltip
        )

        path = os.path.join(HTML_DIR, filename)
        deck.to_html(path, open_browser=False)
        html_paths.append(path)
        print("[PYDECK]", path)

    # 18 3D hidden demand
    hidden_column = pdk.Layer(
        "ColumnLayer",
        data=df,
        get_position="[lon, lat]",
        get_elevation="hidden_demand_score * 50000",
        elevation_scale=1,
        radius=330,
        get_fill_color="hidden_color",
        pickable=True,
        auto_highlight=True,
    )

    save_deck(
        [hidden_column],
        "18_pydeck_hidden_demand_3d_column.html",
        pitch=55
    )

    # 19 heatmap
    hidden_heat = pdk.Layer(
        "HeatmapLayer",
        data=df,
        get_position="[lon, lat]",
        get_weight="hidden_demand_score",
        radiusPixels=70,
        intensity=1.2,
        threshold=0.02,
    )

    save_deck(
        [hidden_heat],
        "19_pydeck_hidden_demand_heatmap.html",
        pitch=0
    )

    # 20 mismatch glow
    mismatch_layer = pdk.Layer(
        "ScatterplotLayer",
        data=df,
        get_position="[lon, lat]",
        get_radius="80 + directional_mismatch_score * 22000",
        get_fill_color="mismatch_color",
        get_line_color="[255,255,255,160]",
        line_width_min_pixels=1,
        pickable=True,
        auto_highlight=True,
    )

    save_deck(
        [mismatch_layer],
        "20_pydeck_directional_mismatch_glow.html",
        pitch=35
    )

    # 21 station typology
    type_layer = pdk.Layer(
        "ScatterplotLayer",
        data=df,
        get_position="[lon, lat]",
        get_radius=450,
        get_fill_color="station_type_color",
        get_line_color="[255,255,255,120]",
        line_width_min_pixels=1,
        pickable=True,
        auto_highlight=True,
    )

    save_deck(
        [type_layer],
        "21_pydeck_station_typology.html",
        pitch=35
    )

    # 22 cluster topology
    cluster_layer = pdk.Layer(
        "ScatterplotLayer",
        data=df,
        get_position="[lon, lat]",
        get_radius=430,
        get_fill_color="cluster_color",
        get_line_color="[255,255,255,120]",
        line_width_min_pixels=1,
        pickable=True,
        auto_highlight=True,
    )

    save_deck(
        [cluster_layer],
        "22_pydeck_cluster_topology.html",
        pitch=35
    )

    # 23 hidden + mismatch combined
    save_deck(
        [hidden_column, mismatch_layer],
        "23_pydeck_hidden_plus_mismatch_combined.html",
        pitch=55
    )

except Exception as e:
    print("[SKIP PYDECK]", e)


# ============================================================
# 9. SAVE TOP TABLES
# ============================================================

df.sort_values("hidden_demand_score", ascending=False).head(32).to_csv(
    os.path.join(CSV_DIR, "top32_robust_hidden_stations.csv"),
    index=False,
    encoding="utf-8-sig"
)

df.sort_values("directional_mismatch_score", ascending=False).head(25).to_csv(
    os.path.join(CSV_DIR, "top25_directional_mismatch_stations.csv"),
    index=False,
    encoding="utf-8-sig"
)

df.to_csv(
    os.path.join(CSV_DIR, "station_level_map_ready_with_quadrant.csv"),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 10. INDEX HTML
# ============================================================

html_paths = sorted(list(dict.fromkeys(html_paths)))

index_path = os.path.join(OUT_DIR, "OPEN_ME_INDEX.html")

with open(index_path, "w", encoding="utf-8") as f:
    f.write("""
    <html>
    <head>
    <meta charset="utf-8">
    <title>Station Mobility Map Viewer</title>
    <style>
    body {
        background: #111;
        color: #eee;
        font-family: Arial, sans-serif;
        padding: 28px;
    }
    a {
        color: #7dd3fc;
        font-size: 17px;
        text-decoration: none;
    }
    .card {
        border: 1px solid #333;
        border-radius: 10px;
        background: #1b1b1b;
        padding: 14px;
        margin: 10px 0;
    }
    .small {
        color: #aaa;
        font-size: 13px;
    }
    </style>
    </head>
    <body>
    <h1>Station Mobility Map Viewer</h1>
    <p class="small">
    Station-level visualization for 327 subway stations.
    Files generated from the station-level compare result, not cluster summary.
    </p>
    """)

    for p in html_paths:
        rel = os.path.relpath(p, OUT_DIR).replace("\\", "/")
        f.write(f"""
        <div class="card">
            <a href="{rel}" target="_blank">{os.path.basename(p)}</a>
        </div>
        """)

    f.write("""
    </body>
    </html>
    """)

print("\nDONE")
print("[SELECTED INPUT]", INPUT_PATH)
print("[OUTPUT ROOT]", OUT_DIR)
print("[OPEN THIS]", index_path)

webbrowser.open(index_path)


[CANDIDATE FILES]
200 (327, 47) C:\Users\82108\Desktop\새 폴더 (8)\diverse_mobility_subway_analysis_filtered\08_compare_subway_total_filtered.csv
170 (327, 47) C:\Users\82108\Desktop\새 폴더 (8)\diverse_mobility_subway_analysis_filtered\08_compare_subway_30min_filtered.csv
170 (327, 47) C:\Users\82108\Desktop\새 폴더 (8)\diverse_mobility_subway_analysis_filtered\08_compare_subway_daily_filtered.csv
50 (116, 48) C:\Users\82108\Desktop\새 폴더 (8)\advanced_station_analysis\02_cluster_representative_stations.csv
50 (35, 48) C:\Users\82108\Desktop\새 폴더 (8)\advanced_station_analysis\03_hidden_demand_corridor.csv
50 (6, 48) C:\Users\82108\Desktop\새 폴더 (8)\advanced_station_analysis\06_mobility_rich_subway_poor.csv
50 (5, 48) C:\Users\82108\Desktop\새 폴더 (8)\advanced_station_analysis\07_subway_rich_mobility_poor.csv
50 (100, 49) C:\Users\82108\Desktop\새 폴더 (8)\advanced_station_analysis\08_unusual_station_patterns.csv

[SELECTED INPUT]
C:\Users\82108\Desktop\새 폴더 (8)\diverse_mobility_subway_analysis_filter

True

In [ ]:
# ============================================================
# LIGHT ALL-IN-ONE MAP VIS
# uses: pandas, numpy, pyproj, plotly, folium, pydeck
# skips missing packages automatically
# ============================================================

import os, glob, webbrowser, warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

BASE_DIR = r"C:\Users\82108\Desktop\새 폴더 (8)\diverse_mobility_subway_analysis_filtered"

OUT_DIR = os.path.join(BASE_DIR, "LIGHT_ALL_MAP_OUTPUT")
HTML_DIR = os.path.join(OUT_DIR, "html")
CSV_DIR = os.path.join(OUT_DIR, "csv")

os.makedirs(HTML_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

# ============================================================
# 1. CSV 자동 탐색
# ============================================================

csvs = glob.glob(os.path.join(BASE_DIR, "*.csv"))

priority = [
    "advanced_station_analysis",
    "subway_total",
    "compare",
    "station_analysis",
]

INPUT_PATH = None
for key in priority:
    for p in csvs:
        if key.lower() in os.path.basename(p).lower():
            INPUT_PATH = p
            break
    if INPUT_PATH:
        break

if INPUT_PATH is None:
    if len(csvs) == 0:
        raise FileNotFoundError("BASE_DIR 안에 csv가 없음")
    INPUT_PATH = csvs[0]

print("[INPUT]", INPUT_PATH)

df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")
print("[DF]", df.shape)


# ============================================================
# 2. 좌표 확보
# ============================================================

def find_mapping_file():
    parent = os.path.dirname(BASE_DIR)
    patterns = [
        os.path.join(parent, "**", "cell_to_nearest_station_VALID_5km_MASTER_ID_FILTERED.csv"),
        os.path.join(parent, "**", "cell_to_nearest_station_VALID_5km_MASTER_ID.csv"),
        os.path.join(parent, "**", "*station*mapping*.csv"),
        os.path.join(parent, "**", "*nearest_station*.csv"),
    ]

    found = []
    for pat in patterns:
        found.extend(glob.glob(pat, recursive=True))

    found = list(dict.fromkeys(found))
    return found[0] if found else None


def ensure_lon_lat(df):
    df = df.copy()

    if {"lon", "lat"}.issubset(df.columns):
        print("[COORD] lon/lat already exists")
        return df

    if {"longitude", "latitude"}.issubset(df.columns):
        print("[COORD] longitude/latitude found")
        df["lon"] = df["longitude"]
        df["lat"] = df["latitude"]
        return df

    try:
        from pyproj import Transformer
    except Exception as e:
        raise ImportError("pyproj가 필요함. pip install pyproj 실행") from e

    if {"station_x_5179", "station_y_5179"}.issubset(df.columns):
        print("[COORD] station_x_5179/station_y_5179 found")
        transformer = Transformer.from_crs("EPSG:5179", "EPSG:4326", always_xy=True)
        lon, lat = transformer.transform(
            df["station_x_5179"].values,
            df["station_y_5179"].values
        )
        df["lon"] = lon
        df["lat"] = lat
        return df

    mapping_path = find_mapping_file()
    if mapping_path is None:
        raise ValueError("좌표 없음. lon/lat 또는 station_x_5179/station_y_5179가 필요함")

    print("[MAPPING FOUND]", mapping_path)

    try:
        mp = pd.read_csv(mapping_path, encoding="utf-8-sig")
    except:
        mp = pd.read_csv(mapping_path, encoding="cp949")

    if {"MASTER_STATION_ID", "station_x_5179", "station_y_5179"}.issubset(mp.columns):
        coord = (
            mp.groupby("MASTER_STATION_ID", as_index=False)
            [["station_x_5179", "station_y_5179"]]
            .mean()
        )
        df = df.merge(coord, on="MASTER_STATION_ID", how="left")

    elif {"nearest_station", "station_x_5179", "station_y_5179"}.issubset(mp.columns):
        coord = (
            mp.groupby("nearest_station", as_index=False)
            [["station_x_5179", "station_y_5179"]]
            .mean()
        )
        df = df.merge(coord, on="nearest_station", how="left")

    else:
        raise ValueError("mapping file에 station_x_5179/station_y_5179 없음")

    transformer = Transformer.from_crs("EPSG:5179", "EPSG:4326", always_xy=True)
    lon, lat = transformer.transform(
        df["station_x_5179"].values,
        df["station_y_5179"].values
    )

    df["lon"] = lon
    df["lat"] = lat

    return df


df = ensure_lon_lat(df)
df = df.dropna(subset=["lon", "lat"]).copy()

print("[COORD READY]", df[["nearest_station", "lon", "lat"]].head())


# ============================================================
# 3. 컬럼 보정
# ============================================================

default_cols = [
    "hidden_demand_score",
    "outer_hidden_demand_score",
    "diverse_living_area_score",
    "directional_mismatch_score",
    "mobility_minus_subway_gap",
    "mobility_norm",
    "subway_norm",
    "mean_temporal_entropy",
    "distance_band_entropy",
    "outer_1km_plus_share",
]

for c in default_cols:
    if c not in df.columns:
        df[c] = 0
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

if "cluster" not in df.columns:
    df["cluster"] = -1

if "station_type" not in df.columns:
    df["station_type"] = "unknown"

if "nearest_line" not in df.columns:
    df["nearest_line"] = "unknown"

df["cluster_str"] = df["cluster"].astype(str)
df["station_label"] = df["nearest_station"].astype(str) + " / " + df["nearest_line"].astype(str)

df["hidden_size"] = 5 + df["hidden_demand_score"] * 35
df["mismatch_size"] = 5 + df["directional_mismatch_score"] * 120
df["outer_size"] = 5 + df["outer_hidden_demand_score"] * 80
df["diverse_size"] = 5 + df["diverse_living_area_score"] * 80

df.to_csv(os.path.join(CSV_DIR, "station_map_ready.csv"), index=False, encoding="utf-8-sig")


# ============================================================
# 4. PLOTLY HTML
# ============================================================

html_paths = []

try:
    import plotly.express as px
    import plotly.graph_objects as go

    def save_plotly(fig, filename):
        path = os.path.join(HTML_DIR, filename)
        fig.write_html(path)
        html_paths.append(path)
        print("[PLOTLY]", path)

    common_hover = {
        "nearest_line": True,
        "station_type": True,
        "cluster": True,
        "hidden_demand_score": ":.3f",
        "directional_mismatch_score": ":.3f",
        "outer_hidden_demand_score": ":.3f",
        "diverse_living_area_score": ":.3f",
        "lat": False,
        "lon": False,
    }

    # 1 Hidden demand
    fig = px.scatter_map(
        df, lat="lat", lon="lon",
        size="hidden_demand_score",
        color="hidden_demand_score",
        hover_name="nearest_station",
        hover_data=common_hover,
        zoom=10,
        height=850,
        title="Hidden Demand Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "01_plotly_hidden_demand.html")

    # 2 Directional mismatch
    fig = px.scatter_map(
        df, lat="lat", lon="lon",
        size="directional_mismatch_score",
        color="directional_mismatch_score",
        hover_name="nearest_station",
        hover_data=common_hover,
        zoom=10,
        height=850,
        title="Directional Mismatch Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "02_plotly_directional_mismatch.html")

    # 3 Station type
    fig = px.scatter_map(
        df, lat="lat", lon="lon",
        size="hidden_demand_score",
        color="station_type",
        hover_name="nearest_station",
        hover_data=common_hover,
        zoom=10,
        height=850,
        title="Station Type Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "03_plotly_station_type.html")

    # 4 Cluster
    fig = px.scatter_map(
        df, lat="lat", lon="lon",
        size="hidden_demand_score",
        color="cluster_str",
        hover_name="nearest_station",
        hover_data=common_hover,
        zoom=10,
        height=850,
        title="Cluster Topology Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "04_plotly_cluster_topology.html")

    # 5 Outer hidden demand
    fig = px.scatter_map(
        df, lat="lat", lon="lon",
        size="outer_hidden_demand_score",
        color="outer_hidden_demand_score",
        hover_name="nearest_station",
        hover_data=common_hover,
        zoom=10,
        height=850,
        title="Outer Hidden Demand Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "05_plotly_outer_hidden_demand.html")

    # 6 Diverse living area
    fig = px.scatter_map(
        df, lat="lat", lon="lon",
        size="diverse_living_area_score",
        color="diverse_living_area_score",
        hover_name="nearest_station",
        hover_data=common_hover,
        zoom=10,
        height=850,
        title="Diverse Living Area Hidden Demand Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "06_plotly_diverse_living_area.html")

    # 7 Mobility minus subway gap
    fig = px.scatter_map(
        df, lat="lat", lon="lon",
        size=np.abs(df["mobility_minus_subway_gap"]) + 0.001,
        color="mobility_minus_subway_gap",
        hover_name="nearest_station",
        hover_data=common_hover,
        zoom=10,
        height=850,
        title="Mobility-Subway Divergence Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "07_plotly_mobility_subway_gap.html")

    # 8 Temporal entropy
    fig = px.scatter_map(
        df, lat="lat", lon="lon",
        size="mean_temporal_entropy",
        color="mean_temporal_entropy",
        hover_name="nearest_station",
        hover_data=common_hover,
        zoom=10,
        height=850,
        title="Temporal Entropy Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "08_plotly_temporal_entropy.html")

    # 9 Distance entropy
    fig = px.scatter_map(
        df, lat="lat", lon="lon",
        size="distance_band_entropy",
        color="distance_band_entropy",
        hover_name="nearest_station",
        hover_data=common_hover,
        zoom=10,
        height=850,
        title="Distance-band Entropy Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "09_plotly_distance_entropy.html")

    # 10 Hidden × mismatch quadrant
    hq = df["hidden_demand_score"].quantile(0.75)
    mq = df["directional_mismatch_score"].quantile(0.75)

    df["quadrant"] = "balanced"
    df.loc[(df["hidden_demand_score"] >= hq) & (df["directional_mismatch_score"] < mq), "quadrant"] = "hidden only"
    df.loc[(df["hidden_demand_score"] < hq) & (df["directional_mismatch_score"] >= mq), "quadrant"] = "mismatch only"
    df.loc[(df["hidden_demand_score"] >= hq) & (df["directional_mismatch_score"] >= mq), "quadrant"] = "hidden + mismatch"

    fig = px.scatter_map(
        df, lat="lat", lon="lon",
        size="hidden_demand_score",
        color="quadrant",
        hover_name="nearest_station",
        hover_data=common_hover,
        zoom=10,
        height=850,
        title="Hidden Demand × Directional Mismatch Quadrant Map"
    )
    fig.update_layout(map_style="carto-darkmatter")
    save_plotly(fig, "10_plotly_hidden_mismatch_quadrant.html")

except Exception as e:
    print("[SKIP PLOTLY]", e)


# ============================================================
# 5. FOLIUM HTML
# ============================================================

try:
    import folium
    from folium.plugins import HeatMap, MarkerCluster

    center = [float(df["lat"].mean()), float(df["lon"].mean())]

    def save_folium(m, filename):
        path = os.path.join(HTML_DIR, filename)
        m.save(path)
        html_paths.append(path)
        print("[FOLIUM]", path)

    # 11 Satellite hidden demand
    m = folium.Map(location=center, zoom_start=11, tiles=None)

    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Imagery",
        name="Satellite",
        overlay=False,
        control=True
    ).add_to(m)

    folium.TileLayer("CartoDB dark_matter", name="Dark").add_to(m)
    folium.TileLayer("CartoDB positron", name="Light").add_to(m)

    for _, r in df.iterrows():
        if r["hidden_demand_score"] >= 0.60:
            color = "red"
        elif r["hidden_demand_score"] >= 0.45:
            color = "orange"
        elif r["hidden_demand_score"] >= 0.30:
            color = "yellow"
        else:
            color = "blue"

        folium.CircleMarker(
            location=[r["lat"], r["lon"]],
            radius=4 + r["hidden_demand_score"] * 16,
            color=color,
            fill=True,
            fill_opacity=0.65,
            popup=f"""
            <b>{r['nearest_station']}</b><br>
            Line: {r['nearest_line']}<br>
            Type: {r['station_type']}<br>
            Cluster: {r['cluster']}<br>
            Hidden: {r['hidden_demand_score']:.3f}<br>
            Mismatch: {r['directional_mismatch_score']:.3f}
            """
        ).add_to(m)

    folium.LayerControl().add_to(m)
    save_folium(m, "11_folium_satellite_hidden_demand.html")

    # 12 Heatmap hidden
    m = folium.Map(location=center, zoom_start=11, tiles="CartoDB dark_matter")
    heat_data = df[["lat", "lon", "hidden_demand_score"]].values.tolist()
    HeatMap(heat_data, radius=25, blur=18, min_opacity=0.25).add_to(m)
    save_folium(m, "12_folium_hidden_heatmap.html")

    # 13 Directional mismatch
    m = folium.Map(location=center, zoom_start=11, tiles="CartoDB dark_matter")

    for _, r in df.iterrows():
        folium.CircleMarker(
            location=[r["lat"], r["lon"]],
            radius=3 + r["directional_mismatch_score"] * 45,
            color="cyan",
            fill=True,
            fill_opacity=0.65,
            popup=f"""
            <b>{r['nearest_station']}</b><br>
            Line: {r['nearest_line']}<br>
            Mismatch: {r['directional_mismatch_score']:.3f}<br>
            Hidden: {r['hidden_demand_score']:.3f}
            """
        ).add_to(m)

    save_folium(m, "13_folium_directional_mismatch.html")

    # 14 Marker cluster
    m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")
    mc = MarkerCluster().add_to(m)

    for _, r in df.iterrows():
        folium.Marker(
            location=[r["lat"], r["lon"]],
            popup=f"""
            <b>{r['nearest_station']}</b><br>
            Line: {r['nearest_line']}<br>
            Type: {r['station_type']}<br>
            Hidden: {r['hidden_demand_score']:.3f}
            """
        ).add_to(mc)

    save_folium(m, "14_folium_marker_cluster.html")

    # 15 Top robust hidden only
    top = df.sort_values("hidden_demand_score", ascending=False).head(32)

    m = folium.Map(location=center, zoom_start=11, tiles=None)

    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Imagery",
        name="Satellite",
        overlay=False,
        control=True
    ).add_to(m)

    for _, r in top.iterrows():
        folium.CircleMarker(
            location=[r["lat"], r["lon"]],
            radius=7 + r["hidden_demand_score"] * 18,
            color="red",
            fill=True,
            fill_opacity=0.75,
            popup=f"""
            <b>{r['nearest_station']}</b><br>
            Line: {r['nearest_line']}<br>
            Hidden: {r['hidden_demand_score']:.3f}<br>
            Outer: {r['outer_hidden_demand_score']:.3f}<br>
            Diverse: {r['diverse_living_area_score']:.3f}<br>
            Mismatch: {r['directional_mismatch_score']:.3f}
            """
        ).add_to(m)

        folium.Marker(
            location=[r["lat"], r["lon"]],
            icon=folium.DivIcon(
                html=f"""
                <div style="
                    font-size:11px;
                    color:white;
                    background:rgba(0,0,0,0.55);
                    padding:2px 4px;
                    border-radius:4px;">
                    {r['nearest_station']}
                </div>
                """
            )
        ).add_to(m)

    save_folium(m, "15_folium_robust_hidden_labeled_satellite.html")

except Exception as e:
    print("[SKIP FOLIUM]", e)


# ============================================================
# 6. PYDECK HTML
# ============================================================

try:
    import pydeck as pdk

    def rgba_hidden(v):
        if v >= 0.60:
            return [255, 0, 0, 230]
        elif v >= 0.45:
            return [255, 100, 0, 220]
        elif v >= 0.30:
            return [255, 220, 0, 200]
        else:
            return [80, 160, 255, 130]

    def rgba_mismatch(v):
        if v >= 0.15:
            return [0, 255, 255, 235]
        elif v >= 0.08:
            return [0, 180, 255, 220]
        elif v >= 0.04:
            return [80, 120, 255, 190]
        else:
            return [180, 180, 180, 90]

    cluster_palette = {
        0: [230, 25, 75, 220],
        1: [60, 180, 75, 220],
        2: [0, 130, 200, 220],
        3: [245, 130, 48, 220],
        4: [145, 30, 180, 220],
        5: [70, 240, 240, 220],
        -1: [180, 180, 180, 180],
    }

    station_type_palette = {
        "normal": [170, 170, 170, 120],
        "directional_mismatch": [0, 255, 255, 235],
        "subway_high_mobility_low": [70, 130, 255, 220],
        "diverse_living_area_hidden": [255, 80, 0, 235],
        "outer_hidden_demand": [255, 0, 180, 235],
        "mobility_high_subway_low": [255, 220, 0, 235],
        "unknown": [180, 180, 180, 150],
    }

    df["hidden_color"] = df["hidden_demand_score"].apply(rgba_hidden)
    df["mismatch_color"] = df["directional_mismatch_score"].apply(rgba_mismatch)
    df["cluster_color"] = df["cluster"].apply(lambda x: cluster_palette.get(int(x), [180,180,180,180]))
    df["station_type_color"] = df["station_type"].apply(lambda x: station_type_palette.get(str(x), [180,180,180,150]))

    tooltip = {
        "html": """
        <b>{nearest_station}</b><br/>
        Line: {nearest_line}<br/>
        Type: {station_type}<br/>
        Cluster: {cluster}<br/>
        Hidden: {hidden_demand_score}<br/>
        Mismatch: {directional_mismatch_score}<br/>
        Outer: {outer_hidden_demand_score}<br/>
        Diverse: {diverse_living_area_score}
        """,
        "style": {"backgroundColor": "black", "color": "white"}
    }

    view = pdk.ViewState(
        latitude=float(df["lat"].mean()),
        longitude=float(df["lon"].mean()),
        zoom=10.2,
        pitch=50,
        bearing=0
    )

    def save_deck(layers, filename, pitch=50):
        deck = pdk.Deck(
            layers=layers,
            initial_view_state=pdk.ViewState(
                latitude=float(df["lat"].mean()),
                longitude=float(df["lon"].mean()),
                zoom=10.2,
                pitch=pitch,
                bearing=0
            ),
            map_style=pdk.map_styles.CARTO_DARK,
            tooltip=tooltip
        )
        path = os.path.join(HTML_DIR, filename)
        deck.to_html(path, open_browser=False)
        html_paths.append(path)
        print("[PYDECK]", path)

    # 16 3D hidden
    hidden_column = pdk.Layer(
        "ColumnLayer",
        data=df,
        get_position="[lon, lat]",
        get_elevation="hidden_demand_score * 50000",
        elevation_scale=1,
        radius=330,
        get_fill_color="hidden_color",
        pickable=True,
        auto_highlight=True,
    )
    save_deck([hidden_column], "16_pydeck_hidden_3d_column.html", pitch=55)

    # 17 heatmap
    hidden_heat = pdk.Layer(
        "HeatmapLayer",
        data=df,
        get_position="[lon, lat]",
        get_weight="hidden_demand_score",
        radiusPixels=70,
        intensity=1.2,
        threshold=0.02,
    )
    save_deck([hidden_heat], "17_pydeck_hidden_heatmap.html", pitch=0)

    # 18 mismatch glow
    mismatch_layer = pdk.Layer(
        "ScatterplotLayer",
        data=df,
        get_position="[lon, lat]",
        get_radius="80 + directional_mismatch_score * 22000",
        get_fill_color="mismatch_color",
        get_line_color="[255,255,255,160]",
        line_width_min_pixels=1,
        pickable=True,
        auto_highlight=True,
    )
    save_deck([mismatch_layer], "18_pydeck_directional_mismatch_glow.html", pitch=35)

    # 19 typology
    type_layer = pdk.Layer(
        "ScatterplotLayer",
        data=df,
        get_position="[lon, lat]",
        get_radius=450,
        get_fill_color="station_type_color",
        get_line_color="[255,255,255,120]",
        line_width_min_pixels=1,
        pickable=True,
        auto_highlight=True,
    )
    save_deck([type_layer], "19_pydeck_station_typology.html", pitch=35)

    # 20 cluster
    cluster_layer = pdk.Layer(
        "ScatterplotLayer",
        data=df,
        get_position="[lon, lat]",
        get_radius=430,
        get_fill_color="cluster_color",
        get_line_color="[255,255,255,120]",
        line_width_min_pixels=1,
        pickable=True,
        auto_highlight=True,
    )
    save_deck([cluster_layer], "20_pydeck_cluster_topology.html", pitch=35)

    # 21 combined
    save_deck([hidden_column, mismatch_layer], "21_pydeck_hidden_plus_mismatch.html", pitch=55)

    # 22 inferred corridor
    try:
        from sklearn.neighbors import NearestNeighbors

        topN = df.sort_values("hidden_demand_score", ascending=False).head(40).copy()
        coords = topN[["lon", "lat"]].values

        if len(topN) >= 4:
            nbrs = NearestNeighbors(n_neighbors=min(4, len(topN))).fit(coords)
            distances, indices = nbrs.kneighbors(coords)

            edges = []
            for i, row in enumerate(indices):
                for j in row[1:]:
                    a = topN.iloc[i]
                    b = topN.iloc[j]
                    strength = float((a["hidden_demand_score"] + b["hidden_demand_score"]) / 2)
                    edges.append({
                        "source": a["nearest_station"],
                        "target": b["nearest_station"],
                        "source_lon": a["lon"],
                        "source_lat": a["lat"],
                        "target_lon": b["lon"],
                        "target_lat": b["lat"],
                        "strength": strength,
                    })

            edge_df = pd.DataFrame(edges).drop_duplicates(subset=["source", "target"])
            edge_df.to_csv(
                os.path.join(CSV_DIR, "inferred_hidden_corridors_top40.csv"),
                index=False,
                encoding="utf-8-sig"
            )

            arc_layer = pdk.Layer(
                "ArcLayer",
                data=edge_df,
                get_source_position="[source_lon, source_lat]",
                get_target_position="[target_lon, target_lat]",
                get_source_color=[255, 80, 0, 180],
                get_target_color=[0, 180, 255, 180],
                get_width="strength * 8",
                pickable=True,
                auto_highlight=True,
            )

            save_deck([arc_layer, hidden_column], "22_pydeck_inferred_hidden_corridors.html", pitch=55)

    except Exception as e:
        print("[SKIP PYDECK CORRIDOR]", e)

except Exception as e:
    print("[SKIP PYDECK]", e)


# ============================================================
# 7. INDEX HTML
# ============================================================

html_paths = sorted(list(dict.fromkeys(html_paths)))

index_path = os.path.join(OUT_DIR, "OPEN_ME_INDEX.html")

with open(index_path, "w", encoding="utf-8") as f:
    f.write("""
    <html>
    <head>
    <meta charset="utf-8">
    <title>Light Urban Mobility Map Viewer</title>
    <style>
    body {
        background: #111;
        color: #eee;
        font-family: Arial, sans-serif;
        padding: 28px;
    }
    a {
        color: #7dd3fc;
        font-size: 17px;
        text-decoration: none;
    }
    .card {
        border: 1px solid #333;
        border-radius: 10px;
        background: #1b1b1b;
        padding: 14px;
        margin: 10px 0;
    }
    .small {
        color: #aaa;
        font-size: 13px;
    }
    </style>
    </head>
    <body>
    <h1>Light Urban Mobility Map Viewer</h1>
    <p class="small">Generated without geopandas, contextily, or keplergl.</p>
    """)

    for p in html_paths:
        rel = os.path.relpath(p, OUT_DIR).replace("\\", "/")
        f.write(f"""
        <div class="card">
            <a href="{rel}" target="_blank">{os.path.basename(p)}</a>
        </div>
        """)

    f.write("""
    </body>
    </html>
    """)

print("\nDONE")
print("[OUTPUT ROOT]", OUT_DIR)
print("[OPEN THIS]", index_path)

webbrowser.open(index_path)

[INPUT] C:\Users\82108\Desktop\새 폴더 (8)\advanced_station_analysis\01_cluster_summary.csv
[DF] (6, 13)
[MAPPING FOUND] C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km_MASTER_ID_FILTERED.csv


KeyError: 'MASTER_STATION_ID'